# ResNet50 Deep Features with SVM and Random Forest

This notebook implements hybrid brain tumour MRI classification using fixed ImageNet-pretrained ResNet50 deep features with Support Vector Machine and Random Forest classifiers, evaluated using the predefined five-fold cross-validation assignments.

In [1]:
# ============================================================
# ResNet50 hybrid experiment - environment check
# ============================================================

import sys
import tensorflow as tf
import sklearn

print("Python version: ", sys.version)
print("TensorFlow version: ", tf.__version__)
print("Keras version: ", tf.keras.__version__)
print("Scikit-Learn version: ", sklearn.__version__)

gpu_devices = tf.config.list_physical_devices("GPU")

if gpu_devices:
    print("GPU is available")
else:
    print("No GPU is available")

!nvidia-smi --query-gpu=name --format=csv,noheader



Python version:  3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
TensorFlow version:  2.20.0
Keras version:  3.13.2
Scikit-Learn version:  1.6.1
No GPU is available
/bin/bash: line 1: nvidia-smi: command not found


In [1]:
# Accessing the google drive for the data

from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
from google.colab import files

uploaded = files.upload()

Saving repeated_stratified_5fold_assignments.csv to repeated_stratified_5fold_assignments.csv


In [3]:
import shutil
import tarfile

from pathlib import Path


# ============================================================
# RESTORE THE PROJECT DATASET TO THE COLAB RUNTIME
# ============================================================

# Create the path to the project archive stored in Google Drive
DRIVE_ARCHIVE = Path("/content/drive/MyDrive/brain_tumour_colab/brain_tumour_colab_bundle.tar")


# Create the local path where the archive will temporarily be copied
LOCAL_ARCHIVE = Path("/content/brain_tumour_colab_bundle.tar")


# Create the path where the project will be restored
PROJECT_ROOT = Path("/content/brain-tumour-mri-classification")


print("=" * 70)
print("RESTORE PROJECT DATA")
print("=" * 70)
print("Drive archive:", DRIVE_ARCHIVE)
print("Drive archive exists:", DRIVE_ARCHIVE.exists())
print("Project root:", PROJECT_ROOT)


# Stop if the project archive cannot be found in Google Drive
if not DRIVE_ARCHIVE.exists():
    raise FileNotFoundError(f"Archive not found: {DRIVE_ARCHIVE}")


# Restore the project only if it is not already present in the runtime
if not PROJECT_ROOT.exists():

    print("\nCopying archive to Colab runtime...")
    shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)


    # Create the project directory
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


    print("Extracting archive...")

    with tarfile.open(LOCAL_ARCHIVE, "r") as archive:
        archive.extractall(PROJECT_ROOT, filter="data")

else:
    print("\nProject directory already exists. Skipping archive extraction.")


# ============================================================
# VERIFY THE RESTORED PROJECT
# ============================================================

# Create the path to the fixed five-fold cross-validation file
FOLDS_FILE = (PROJECT_ROOT / "splits" / "five_fold_cross_validation.csv")


# Create the path to the cropped Training and Testing dataset
DATA_DIR = (PROJECT_ROOT / "processed_data_cropped")


# Count every PNG image in the cropped dataset
png_image_count = len(list(DATA_DIR.rglob("*.png")))

print("\n--- Verification ---")
print("Project root exists:", PROJECT_ROOT.exists())
print("Fold file exists:", FOLDS_FILE.exists())
print("Dataset folder exists:", DATA_DIR.exists())
print("PNG images found:", png_image_count)


# Stop if the project was not restored correctly
if not FOLDS_FILE.exists():
    raise FileNotFoundError(f"Five-fold cross-validation file not found: {FOLDS_FILE}")


if not DATA_DIR.exists():
    raise FileNotFoundError(f"Cropped dataset folder not found: {DATA_DIR}")


if png_image_count != 7198:
    raise RuntimeError("Expected 7,198 PNG images, "f"but found {png_image_count}.")

print("\nProject restored successfully.")

RESTORE PROJECT DATA
Drive archive: /content/drive/MyDrive/brain_tumour_colab/brain_tumour_colab_bundle.tar
Drive archive exists: True
Project root: /content/brain-tumour-mri-classification

Copying archive to Colab runtime...
Extracting archive...

--- Verification ---
Project root exists: True
Fold file exists: True
Dataset folder exists: True
PNG images found: 7198

Project restored successfully.


In [4]:
# ============================================================
# VALIDATE THE REPEATED 10 x 5 CROSS-VALIDATION ASSIGNMENTS
# ============================================================

from pathlib import Path
import shutil
import pandas as pd


UPLOADED_SPLIT_FILE = Path(
    "/content/repeated_stratified_5fold_assignments.csv"
)

REPEATED_SPLITS_DIR = (
    PROJECT_ROOT
    / "repeated_cv"
    / "splits"
)

REPEATED_SPLITS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPEATED_FOLDS_FILE = (
    REPEATED_SPLITS_DIR
    / "repeated_stratified_5fold_assignments.csv"
)


# Copy the uploaded file into the restored project structure
shutil.copy2(
    UPLOADED_SPLIT_FILE,
    REPEATED_FOLDS_FILE,
)


# Load the assignments
repeated_folds_df = pd.read_csv(
    REPEATED_FOLDS_FILE
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(repeated_folds_df) == 56000

assert repeated_folds_df["repeat"].nunique() == 10

assert set(
    repeated_folds_df["repeat"]
) == set(range(1, 11))

assert set(
    repeated_folds_df["fold"]
) == {1, 2, 3, 4, 5}

assert (
    repeated_folds_df
    .groupby(
        ["repeat", "fold"]
    )
    .size()
    .eq(1120)
    .all()
)

assert (
    repeated_folds_df
    .groupby(
        ["repeat", "fold", "class"]
    )
    .size()
    .eq(280)
    .all()
)

assert (
    repeated_folds_df[
        "relative_path"
    ]
    .str.startswith("Training/")
    .all()
)


print(
    "Repeated split file:",
    REPEATED_FOLDS_FILE,
)

print(
    "Rows:",
    len(repeated_folds_df),
)

print(
    "Repeats:",
    repeated_folds_df[
        "repeat"
    ].nunique(),
)

print(
    "Folds per repeat:",
    repeated_folds_df[
        "fold"
    ].nunique(),
)

print(
    "\nSplit seeds:"
)

print(
    repeated_folds_df[
        ["repeat", "split_seed"]
    ]
    .drop_duplicates()
    .sort_values("repeat")
    .to_string(index=False)
)

print(
    "\nRepeated 10 x 5 split validation PASSED."
)

Repeated split file: /content/brain-tumour-mri-classification/repeated_cv/splits/repeated_stratified_5fold_assignments.csv
Rows: 56000
Repeats: 10
Folds per repeat: 5

Split seeds:
 repeat  split_seed
      1      202601
      2      202602
      3      202603
      4      202604
      5      202605
      6      202606
      7      202607
      8      202608
      9      202609
     10      202610

Repeated 10 x 5 split validation PASSED.


In [5]:
# ============================================================
# Step 5. REPEATED NESTED-CV RESNET50 FIXED-FEATURE
# HYBRID EXPERIMENT SETUP
#
# IMPORTANT:
#   - Uses the EXISTING ResNet50 feature cache created during
#     the original Hybrid experiment.
#   - ResNet50 features are NOT extracted again.
#   - Old cached five-fold assignments will NOT be used.
#   - The shared repeated 10 × 5 assignments loaded earlier
#     will control all new outer folds.
# ============================================================

import gc
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATA_DIR = (
    PROJECT_ROOT
    / "processed_data_cropped"
)

assert DATA_DIR.exists(), (
    "Dataset directory was not found:\n"
    f"{DATA_DIR}"
)


# ------------------------------------------------------------
# Persistent Google Drive storage
# ------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/brain_tumour_colab"
)


assert DRIVE_ROOT.exists(), (
    "Google Drive experiment directory was not found:\n"
    f"{DRIVE_ROOT}"
)


# ------------------------------------------------------------
# EXISTING ResNet50 fixed-feature cache
#
# This file was produced by the original Hybrid experiment.
# It contains the already-extracted fixed ImageNet ResNet50
# features for all 5,600 Training images.
#
# DO NOT create another feature matrix for the repeated run.
# ------------------------------------------------------------

RESNET50_FEATURE_CACHE_PATH = (
    DRIVE_ROOT
    / "results"
    / "resnet50_fixed_features"
    / "training_resnet50_features.npz"
)


assert RESNET50_FEATURE_CACHE_PATH.exists(), (
    "Existing ResNet50 feature cache was not found:\n"
    f"{RESNET50_FEATURE_CACHE_PATH}\n\n"
    "Do not extract new features until this has been checked."
)


# ------------------------------------------------------------
# Persistent repeated-CV result directories
# ------------------------------------------------------------

REPEATED_RESNET50_SVM_DIR = (
    DRIVE_ROOT
    / "results"
    / "repeated_nested_cv"
    / "resnet50_svm"
)

REPEATED_RESNET50_SVM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


REPEATED_RESNET50_RF_DIR = (
    DRIVE_ROOT
    / "results"
    / "repeated_nested_cv"
    / "resnet50_rf"
)

REPEATED_RESNET50_RF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Image / feature / class configuration
# ------------------------------------------------------------

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

FEATURE_DIMENSION = 2048


CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary",
]


CLASS_TO_INDEX = {
    class_name: index
    for index, class_name
    in enumerate(CLASS_NAMES)
}


INDEX_TO_CLASS = {
    index: class_name
    for class_name, index
    in CLASS_TO_INDEX.items()
}


# ------------------------------------------------------------
# Repeated nested-CV configuration
# ------------------------------------------------------------

NUMBER_OF_REPEATS = 10

NUMBER_OF_OUTER_FOLDS = 5

NUMBER_OF_INNER_FOLDS = 3


# ------------------------------------------------------------
# Deterministic inner-CV seed
# ------------------------------------------------------------

BASE_INNER_CV_SEED = 303000


def get_hybrid_inner_cv_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_INNER_CV_SEED
        + repeat_number * 100
        + fold_number
    )


# ------------------------------------------------------------
# Random Forest model/search seeds
# ------------------------------------------------------------

BASE_RF_MODEL_SEED = 404000

BASE_RF_SEARCH_SEED = 414000


def get_hybrid_rf_model_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_RF_MODEL_SEED
        + repeat_number * 100
        + fold_number
    )


def get_hybrid_rf_search_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_RF_SEARCH_SEED
        + repeat_number * 100
        + fold_number
    )


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

os.environ[
    "PYTHONHASHSEED"
] = str(
    RANDOM_SEED
)

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

tf.keras.utils.set_random_seed(
    RANDOM_SEED
)


try:

    tf.config.experimental.enable_op_determinism()

    determinism_status = "enabled"

except Exception as error:

    determinism_status = (
        f"requested but unavailable: {error}"
    )


tf.keras.backend.set_floatx(
    "float32"
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert len(
    repeated_folds_df
) == 56000


assert (
    repeated_folds_df[
        "repeat"
    ].nunique()
    == NUMBER_OF_REPEATS
)


assert (
    repeated_folds_df[
        "fold"
    ].nunique()
    == NUMBER_OF_OUTER_FOLDS
)


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("=" * 70)

print(
    "REPEATED NESTED-CV RESNET50 FIXED-FEATURE SETUP"
)

print("=" * 70)


print(
    "Dataset:",
    DATA_DIR,
)


print(
    "\nExisting ResNet50 feature cache:",
    RESNET50_FEATURE_CACHE_PATH,
)


print(
    "Feature cache exists:",
    RESNET50_FEATURE_CACHE_PATH.exists(),
)


print(
    "\nResNet50 + SVM results:",
    REPEATED_RESNET50_SVM_DIR,
)


print(
    "ResNet50 + RF results:",
    REPEATED_RESNET50_RF_DIR,
)


print(
    "\nRepeats:",
    NUMBER_OF_REPEATS,
)


print(
    "Outer folds per repeat:",
    NUMBER_OF_OUTER_FOLDS,
)


print(
    "Total outer evaluations per classifier:",
    (
        NUMBER_OF_REPEATS
        * NUMBER_OF_OUTER_FOLDS
    ),
)


print(
    "Inner CV folds:",
    NUMBER_OF_INNER_FOLDS,
)


print(
    "\nExpected feature dimension:",
    FEATURE_DIMENSION,
)


print(
    "Feature extraction required:",
    False,
)


print(
    "\nRepeat 1 / Fold 1 inner-CV seed:",
    get_hybrid_inner_cv_seed(
        1,
        1,
    ),
)


print(
    "Repeat 1 / Fold 1 RF model seed:",
    get_hybrid_rf_model_seed(
        1,
        1,
    ),
)


print(
    "Repeat 1 / Fold 1 RF search seed:",
    get_hybrid_rf_search_seed(
        1,
        1,
    ),
)


print(
    "\nIMPORTANT:"
)

print(
    "The previously extracted ResNet50 feature cache "
    "will be reused."
)

print(
    "No ResNet50 feature extraction will be performed."
)

print(
    "Old cached five-fold assignments will be ignored."
)

print(
    "The shared repeated 10 × 5 assignments will be used."
)

print(
    "The Testing partition is not used."
)


print(
    "\nHybrid Step 5 repeated-CV setup PASSED."
)

REPEATED NESTED-CV RESNET50 FIXED-FEATURE SETUP
Dataset: /content/brain-tumour-mri-classification/processed_data_cropped

Existing ResNet50 feature cache: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/training_resnet50_features.npz
Feature cache exists: True

ResNet50 + SVM results: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50_svm
ResNet50 + RF results: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50_rf

Repeats: 10
Outer folds per repeat: 5
Total outer evaluations per classifier: 50
Inner CV folds: 3

Expected feature dimension: 2048
Feature extraction required: False

Repeat 1 / Fold 1 inner-CV seed: 303101
Repeat 1 / Fold 1 RF model seed: 404101
Repeat 1 / Fold 1 RF search seed: 414101

IMPORTANT:
The previously extracted ResNet50 feature cache will be reused.
No ResNet50 feature extraction will be performed.
Old cached five-fold assignments will be ignored.
The shared repeated 10 × 5 assignme

In [6]:
# ============================================================
# Step 6. LOAD AND VALIDATE THE EXISTING
# RESNET50 FIXED-FEATURE CACHE
#
# IMPORTANT:
#   - Features are loaded from the original Hybrid experiment.
#   - Features are NOT extracted again.
#   - The old cached five-fold assignments are ignored.
#   - Feature rows are aligned to a canonical 5,600-image
#     Training manifest derived from the shared repeated splits.
# ============================================================


# ------------------------------------------------------------
# 1. Load the existing feature archive
# ------------------------------------------------------------

feature_archive = np.load(
    RESNET50_FEATURE_CACHE_PATH,
    allow_pickle=True,
)


print(
    "Feature archive contents:",
    feature_archive.files,
)


# ------------------------------------------------------------
# 2. Required archive contents
# ------------------------------------------------------------

required_cache_arrays = {
    "features",
    "class_indices",
    "relative_paths",
}


missing_cache_arrays = (
    required_cache_arrays
    - set(
        feature_archive.files
    )
)


assert not missing_cache_arrays, (
    "Existing ResNet50 feature cache is missing "
    f"required arrays: {sorted(missing_cache_arrays)}"
)


# ------------------------------------------------------------
# 3. Load the cached arrays
#
# NOTE:
#   fold_assignments may also exist in the archive because
#   it was created during the original five-fold experiment.
#
#   Those old fold assignments are deliberately NOT loaded
#   for use in the repeated experiment.
# ------------------------------------------------------------

cached_resnet50_features = (
    feature_archive[
        "features"
    ]
)

cached_resnet50_class_indices = (
    feature_archive[
        "class_indices"
    ]
    .astype(
        np.int32
    )
)

cached_resnet50_relative_paths = (
    feature_archive[
        "relative_paths"
    ]
    .astype(str)
)


# ------------------------------------------------------------
# 4. Basic feature-cache validation
# ------------------------------------------------------------

assert cached_resnet50_features.shape == (
    5600,
    FEATURE_DIMENSION,
), (
    "Unexpected ResNet50 feature matrix shape: "
    f"{cached_resnet50_features.shape}"
)


assert cached_resnet50_features.dtype == np.float32, (
    "Expected float32 ResNet50 features, but found "
    f"{cached_resnet50_features.dtype}"
)


assert len(
    cached_resnet50_class_indices
) == 5600


assert len(
    cached_resnet50_relative_paths
) == 5600


assert (
    len(
        np.unique(
            cached_resnet50_relative_paths
        )
    )
    == 5600
), (
    "Duplicate image paths were found "
    "in the cached ResNet50 features."
)


assert np.isfinite(
    cached_resnet50_features
).all(), (
    "Non-finite values were found "
    "in the cached ResNet50 feature matrix."
)


assert set(
    np.unique(
        cached_resnet50_class_indices
    )
) == {
    0,
    1,
    2,
    3,
}, (
    "Unexpected class indices were found "
    "in the feature cache."
)


# ------------------------------------------------------------
# 5. Create the canonical Training manifest
#
# Every repeat contains the same 5,600 Training images.
# We use Repeat 1 only to obtain one canonical copy.
#
# Fold membership is NOT taken from Repeat 1 here.
# The 'fold' column is deliberately excluded.
# ------------------------------------------------------------

canonical_training_df = (
    repeated_folds_df[
        repeated_folds_df[
            "repeat"
        ]
        == 1
    ][
        [
            "relative_path",
            "class",
            "label",
        ]
    ]
    .copy()
    .sort_values(
        "relative_path"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    canonical_training_df
) == 5600


assert (
    canonical_training_df[
        "relative_path"
    ]
    .nunique()
    == 5600
)


# ------------------------------------------------------------
# 6. Verify the cache contains exactly the same
#    5,600 Training images as the repeated experiment
# ------------------------------------------------------------

cached_path_set = set(
    cached_resnet50_relative_paths.tolist()
)


repeated_training_path_set = set(
    canonical_training_df[
        "relative_path"
    ]
    .astype(str)
    .tolist()
)


assert (
    cached_path_set
    == repeated_training_path_set
), (
    "The existing ResNet50 feature cache and the "
    "shared repeated-CV Training manifest do not "
    "contain exactly the same 5,600 images."
)


# ------------------------------------------------------------
# 7. Align cached feature rows to the canonical
#    Training manifest order
# ------------------------------------------------------------

path_to_cached_index = {
    path: index
    for index, path
    in enumerate(
        cached_resnet50_relative_paths
    )
}


ordered_cache_indices = np.array(
    [
        path_to_cached_index[
            path
        ]
        for path
        in canonical_training_df[
            "relative_path"
        ].astype(str)
    ],
    dtype=np.int64,
)


training_features = (
    cached_resnet50_features[
        ordered_cache_indices
    ]
)


training_class_indices = (
    cached_resnet50_class_indices[
        ordered_cache_indices
    ]
)


training_relative_paths = (
    cached_resnet50_relative_paths[
        ordered_cache_indices
    ]
)


# ------------------------------------------------------------
# 8. Validate labels after path-based alignment
# ------------------------------------------------------------

expected_class_indices = (
    canonical_training_df[
        "label"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


assert np.array_equal(
    training_class_indices,
    expected_class_indices,
), (
    "Cached ResNet50 class indices do not match "
    "the repeated-CV Training manifest after "
    "path-based alignment."
)


assert np.array_equal(
    training_relative_paths,
    canonical_training_df[
        "relative_path"
    ]
    .astype(str)
    .to_numpy(),
), (
    "Cached ResNet50 feature rows are not aligned "
    "with the canonical Training manifest."
)


# ------------------------------------------------------------
# 9. Final feature-matrix validation
# ------------------------------------------------------------

assert training_features.shape == (
    5600,
    2048,
)


assert training_features.dtype == np.float32


assert len(
    training_class_indices
) == 5600


assert len(
    training_relative_paths
) == 5600


assert np.isfinite(
    training_features
).all()


# ------------------------------------------------------------
# 10. Class-distribution validation
# ------------------------------------------------------------

class_counts = (
    pd.Series(
        training_class_indices
    )
    .value_counts()
    .sort_index()
)


assert (
    class_counts
    == 1400
).all(), (
    "Expected exactly 1,400 Training images "
    "from each class."
)


# ------------------------------------------------------------
# 11. Display
# ------------------------------------------------------------

print("=" * 70)

print(
    "EXISTING RESNET50 FEATURE CACHE VALIDATION"
)

print("=" * 70)


print(
    "Loaded from:",
    RESNET50_FEATURE_CACHE_PATH,
)


print(
    "\nFeature matrix:",
    training_features.shape,
)


print(
    "Feature dtype:",
    training_features.dtype,
)


print(
    "Class indices:",
    training_class_indices.shape,
)


print(
    "Unique Training paths:",
    len(
        np.unique(
            training_relative_paths
        )
    ),
)


print(
    "All features finite:",
    np.isfinite(
        training_features
    ).all(),
)


print(
    "\nClass counts:"
)

for class_index, count in (
    class_counts.items()
):

    print(
        f"{INDEX_TO_CLASS[class_index]:12s}:",
        count,
    )


print(
    "\nIMPORTANT:"
)

print(
    "Existing ResNet50 features were LOADED."
)

print(
    "No ResNet50 feature extraction was performed."
)

print(
    "Old cached five-fold assignments were NOT used."
)

print(
    "Feature rows were aligned using image paths."
)

print(
    "Shared repeated 10 × 5 assignments remain authoritative."
)

print(
    "Testing images were not used."
)


print(
    "\nHybrid Step 6 feature-cache validation PASSED."
)

Feature archive contents: ['features', 'class_indices', 'fold_assignments', 'relative_paths']
EXISTING RESNET50 FEATURE CACHE VALIDATION
Loaded from: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/training_resnet50_features.npz

Feature matrix: (5600, 2048)
Feature dtype: float32
Class indices: (5600,)
Unique Training paths: 5600
All features finite: True

Class counts:
glioma      : 1400
meningioma  : 1400
notumor     : 1400
pituitary   : 1400

IMPORTANT:
Existing ResNet50 features were LOADED.
No ResNet50 feature extraction was performed.
Old cached five-fold assignments were NOT used.
Feature rows were aligned using image paths.
Shared repeated 10 × 5 assignments remain authoritative.
Testing images were not used.

Hybrid Step 6 feature-cache validation PASSED.


In [7]:
# ============================================================
# Step 7. BUILD AND VALIDATE THE REPEATED OUTER PARTITIONS
# FOR RESNET50 FIXED FEATURES
#
# IMPORTANT:
#   - Uses the shared repeated 10 × 5 assignment file.
#   - Uses the already-loaded 5,600 × 2,048 feature matrix.
#   - Does NOT use the old cached five-fold assignments.
#   - Does NOT use the Testing partition.
# ============================================================


# ------------------------------------------------------------
# 1. Map every Training image path to its canonical
#    feature-matrix row
# ------------------------------------------------------------

path_to_feature_index = {
    path: index
    for index, path
    in enumerate(
        training_relative_paths
    )
}


assert len(
    path_to_feature_index
) == 5600


# ------------------------------------------------------------
# 2. Create repeated hybrid assignment table
# ------------------------------------------------------------

hybrid_repeated_assignments = (
    repeated_folds_df
    .copy()
)


hybrid_repeated_assignments[
    "feature_index"
] = (
    hybrid_repeated_assignments[
        "relative_path"
    ]
    .astype(str)
    .map(
        path_to_feature_index
    )
)


# ------------------------------------------------------------
# 3. Validate feature-row mapping
# ------------------------------------------------------------

assert (
    hybrid_repeated_assignments[
        "feature_index"
    ]
    .notna()
    .all()
), (
    "At least one repeated-CV image could not "
    "be matched to the cached ResNet50 features."
)


hybrid_repeated_assignments[
    "feature_index"
] = (
    hybrid_repeated_assignments[
        "feature_index"
    ]
    .astype(
        np.int64
    )
)


assert len(
    hybrid_repeated_assignments
) == 56000


# ------------------------------------------------------------
# 4. Define outer-partition function
# ------------------------------------------------------------

def make_hybrid_outer_partition(
    repeat_number,
    fold_number,
):
    """
    Return one repeated outer-training /
    outer-validation partition using the fixed
    ResNet50 feature matrix.

    Hyperparameter tuning will later be performed
    only on the returned outer-training partition.
    """

    assert (
        1
        <= repeat_number
        <= NUMBER_OF_REPEATS
    )

    assert (
        1
        <= fold_number
        <= NUMBER_OF_OUTER_FOLDS
    )


    # --------------------------------------------------------
    # Select one repeat
    # --------------------------------------------------------

    repeat_df = (
        hybrid_repeated_assignments[
            hybrid_repeated_assignments[
                "repeat"
            ]
            == repeat_number
        ]
        .copy()
    )


    assert len(
        repeat_df
    ) == 5600


    # --------------------------------------------------------
    # Identify outer training / validation rows
    # --------------------------------------------------------

    outer_training_df = (
        repeat_df[
            repeat_df[
                "fold"
            ]
            != fold_number
        ]
        .copy()
    )


    outer_validation_df = (
        repeat_df[
            repeat_df[
                "fold"
            ]
            == fold_number
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Obtain feature-matrix indices
    # --------------------------------------------------------

    outer_training_indices = (
        outer_training_df[
            "feature_index"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    outer_validation_indices = (
        outer_validation_df[
            "feature_index"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    # --------------------------------------------------------
    # Obtain features
    # --------------------------------------------------------

    outer_training_features = (
        training_features[
            outer_training_indices
        ]
    )


    outer_validation_features = (
        training_features[
            outer_validation_indices
        ]
    )


    # --------------------------------------------------------
    # Obtain labels
    # --------------------------------------------------------

    outer_training_labels = (
        training_class_indices[
            outer_training_indices
        ]
    )


    outer_validation_labels = (
        training_class_indices[
            outer_validation_indices
        ]
    )


    # --------------------------------------------------------
    # Obtain paths
    # --------------------------------------------------------

    outer_training_paths = (
        training_relative_paths[
            outer_training_indices
        ]
    )


    outer_validation_paths = (
        training_relative_paths[
            outer_validation_indices
        ]
    )


    # --------------------------------------------------------
    # Validate sizes
    # --------------------------------------------------------

    assert outer_training_features.shape == (
        4480,
        FEATURE_DIMENSION,
    )


    assert outer_validation_features.shape == (
        1120,
        FEATURE_DIMENSION,
    )


    assert len(
        outer_training_labels
    ) == 4480


    assert len(
        outer_validation_labels
    ) == 1120


    # --------------------------------------------------------
    # Validate no overlap
    # --------------------------------------------------------

    assert set(
        outer_training_paths
    ).isdisjoint(
        set(
            outer_validation_paths
        )
    ), (
        "Outer Training and validation paths overlap."
    )


    assert (
        len(
            set(
                outer_training_paths
            )
        )
        == 4480
    )


    assert (
        len(
            set(
                outer_validation_paths
            )
        )
        == 1120
    )


    # --------------------------------------------------------
    # Validate class balance
    # --------------------------------------------------------

    outer_training_class_counts = (
        pd.Series(
            outer_training_labels
        )
        .value_counts()
        .sort_index()
    )


    outer_validation_class_counts = (
        pd.Series(
            outer_validation_labels
        )
        .value_counts()
        .sort_index()
    )


    assert (
        outer_training_class_counts
        == 1120
    ).all(), (
        "Each class must contain exactly "
        "1,120 outer-training images."
    )


    assert (
        outer_validation_class_counts
        == 280
    ).all(), (
        "Each class must contain exactly "
        "280 outer-validation images."
    )


    # --------------------------------------------------------
    # Return partition
    # --------------------------------------------------------

    return {
        "repeat": repeat_number,
        "fold": fold_number,

        "split_seed": int(
            outer_validation_df[
                "split_seed"
            ].iloc[0]
        ),

        "outer_training_features":
            outer_training_features,

        "outer_training_labels":
            outer_training_labels,

        "outer_training_paths":
            outer_training_paths,

        "outer_validation_features":
            outer_validation_features,

        "outer_validation_labels":
            outer_validation_labels,

        "outer_validation_paths":
            outer_validation_paths,
    }


# ------------------------------------------------------------
# 5. Test Repeat 1 / Fold 1
# ------------------------------------------------------------

hybrid_partition_test = (
    make_hybrid_outer_partition(
        repeat_number=1,
        fold_number=1,
    )
)


# ------------------------------------------------------------
# 6. Display validation
# ------------------------------------------------------------

print("=" * 70)

print(
    "RESNET50 FIXED-FEATURE REPEATED OUTER PARTITION"
)

print("=" * 70)


print(
    "Repeat:",
    hybrid_partition_test[
        "repeat"
    ],
)


print(
    "Fold:",
    hybrid_partition_test[
        "fold"
    ],
)


print(
    "Split seed:",
    hybrid_partition_test[
        "split_seed"
    ],
)


print(
    "\nOuter Training features:",
    hybrid_partition_test[
        "outer_training_features"
    ].shape,
)


print(
    "Outer validation features:",
    hybrid_partition_test[
        "outer_validation_features"
    ].shape,
)


print(
    "\nOuter Training class counts:"
)

print(
    pd.Series(
        hybrid_partition_test[
            "outer_training_labels"
        ]
    )
    .value_counts()
    .sort_index()
)


print(
    "\nOuter validation class counts:"
)

print(
    pd.Series(
        hybrid_partition_test[
            "outer_validation_labels"
        ]
    )
    .value_counts()
    .sort_index()
)


print(
    "\nIMPORTANT:"
)

print(
    "Old cached five-fold assignments were NOT used."
)

print(
    "Shared repeated 10 × 5 assignments were used."
)

print(
    "Testing images were not used."
)


print(
    "\nHybrid Step 7 repeated outer-partition "
    "validation PASSED."
)

RESNET50 FIXED-FEATURE REPEATED OUTER PARTITION
Repeat: 1
Fold: 1
Split seed: 202601

Outer Training features: (4480, 2048)
Outer validation features: (1120, 2048)

Outer Training class counts:
0    1120
1    1120
2    1120
3    1120
Name: count, dtype: int64

Outer validation class counts:
0    280
1    280
2    280
3    280
Name: count, dtype: int64

IMPORTANT:
Old cached five-fold assignments were NOT used.
Shared repeated 10 × 5 assignments were used.
Testing images were not used.

Hybrid Step 7 repeated outer-partition validation PASSED.


In [8]:
# ============================================================
# Step 8. DEFINE THE REPEATED NESTED-CV
# RESNET50 FIXED-FEATURE + SVM SEARCH
#
# IMPORTANT:
#   - StandardScaler remains INSIDE the sklearn pipeline.
#   - Hyperparameter selection uses only the 4,480
#     outer-training feature vectors.
#   - Every outer fold receives its own deterministic
#     shuffled 3-fold inner CV.
#   - Outer-validation data is not used for selection.
# ============================================================

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
)

from sklearn.svm import SVC


# ------------------------------------------------------------
# 1. SVM hyperparameter grid
#
# Same search used in the original Hybrid experiment:
#
#   C     = 5 values
#   gamma = 5 values
#
# Total = 25 candidate configurations
# ------------------------------------------------------------

SVM_PARAMETER_GRID = {
    "classifier__C": [
        0.01,
        0.1,
        1,
        10,
        100,
    ],

    "classifier__gamma": [
        "scale",
        1e-5,
        1e-4,
        1e-3,
        1e-2,
    ],
}


# ------------------------------------------------------------
# 2. Base SVM pipeline
#
# StandardScaler MUST stay inside the pipeline so that,
# during inner CV, it is fitted only on each inner-training
# subset and never on its corresponding inner-validation set.
# ------------------------------------------------------------

def build_hybrid_svm_pipeline():

    return Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(),
            ),

            (
                "classifier",
                SVC(
                    kernel="rbf",
                ),
            ),
        ]
    )


# ------------------------------------------------------------
# 3. Build one fold-specific inner search
#
# A fresh StratifiedKFold object is created for every
# repeat / outer-fold combination using the documented
# deterministic seed schedule.
# ------------------------------------------------------------

def build_hybrid_svm_search(
    repeat_number,
    fold_number,
):

    inner_cv_seed = (
        get_hybrid_inner_cv_seed(
            repeat_number,
            fold_number,
        )
    )


    inner_cross_validation = (
        StratifiedKFold(
            n_splits=NUMBER_OF_INNER_FOLDS,
            shuffle=True,
            random_state=inner_cv_seed,
        )
    )


    svm_search = GridSearchCV(
        estimator=build_hybrid_svm_pipeline(),

        param_grid=SVM_PARAMETER_GRID,

        scoring="f1_macro",

        cv=inner_cross_validation,

        n_jobs=-1,

        refit=True,

        return_train_score=False,
    )


    return svm_search


# ------------------------------------------------------------
# 4. Validate the complete search definition
# ------------------------------------------------------------

number_of_svm_configurations = (
    len(
        SVM_PARAMETER_GRID[
            "classifier__C"
        ]
    )
    *
    len(
        SVM_PARAMETER_GRID[
            "classifier__gamma"
        ]
    )
)


assert (
    number_of_svm_configurations
    == 25
)


test_svm_search = (
    build_hybrid_svm_search(
        repeat_number=1,
        fold_number=1,
    )
)


assert (
    test_svm_search.scoring
    == "f1_macro"
)


assert (
    test_svm_search.cv.n_splits
    == 3
)


assert (
    test_svm_search.cv.random_state
    == 303101
)


assert (
    test_svm_search.refit
    is True
)


# ------------------------------------------------------------
# 5. Display
# ------------------------------------------------------------

print("=" * 70)

print(
    "RESNET50 FIXED-FEATURE + SVM SEARCH"
)

print("=" * 70)


print(
    "SVM kernel:",
    "RBF",
)


print(
    "Feature standardisation:",
    "StandardScaler inside Pipeline",
)


print(
    "C values:",
    SVM_PARAMETER_GRID[
        "classifier__C"
    ],
)


print(
    "Gamma values:",
    SVM_PARAMETER_GRID[
        "classifier__gamma"
    ],
)


print(
    "Candidate configurations:",
    number_of_svm_configurations,
)


print(
    "Inner CV folds:",
    NUMBER_OF_INNER_FOLDS,
)


print(
    "Selection metric:",
    test_svm_search.scoring,
)


print(
    "\nRepeat 1 / Fold 1 inner-CV seed:",
    test_svm_search.cv.random_state,
)


print(
    "\nIMPORTANT:"
)

print(
    "A fresh SVM search will be created "
    "for every outer fold."
)

print(
    "Only outer-training features will be used "
    "for hyperparameter selection."
)

print(
    "Outer-validation features remain untouched "
    "until final fold evaluation."
)

print(
    "Testing images are not used."
)


print(
    "\nHybrid Step 8 SVM search definition PASSED."
)

RESNET50 FIXED-FEATURE + SVM SEARCH
SVM kernel: RBF
Feature standardisation: StandardScaler inside Pipeline
C values: [0.01, 0.1, 1, 10, 100]
Gamma values: ['scale', 1e-05, 0.0001, 0.001, 0.01]
Candidate configurations: 25
Inner CV folds: 3
Selection metric: f1_macro

Repeat 1 / Fold 1 inner-CV seed: 303101

IMPORTANT:
A fresh SVM search will be created for every outer fold.
Only outer-training features will be used for hyperparameter selection.
Outer-validation features remain untouched until final fold evaluation.
Testing images are not used.

Hybrid Step 8 SVM search definition PASSED.


In [9]:
# ============================================================
# Step 9. RESNET50 FIXED-FEATURE + SVM
# PERSISTENCE AND RESUME UTILITIES
#
# IMPORTANT:
#   - Every completed outer fold will be saved to Google Drive.
#   - Completed folds can be detected after a disconnect.
#   - Existing completed folds will later be skipped.
#   - Files are written atomically to reduce the risk of
#     partially written results.
# ============================================================

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Convert NumPy objects to ordinary Python objects
#    before writing JSON
# ------------------------------------------------------------

def make_json_serializable(value):

    if isinstance(
        value,
        np.integer,
    ):
        return int(
            value
        )

    if isinstance(
        value,
        np.floating,
    ):
        return float(
            value
        )

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    raise TypeError(
        "Object of type "
        f"{type(value).__name__} "
        "is not JSON serializable."
    )


# ------------------------------------------------------------
# 2. Atomic JSON writer
# ------------------------------------------------------------

def save_json_atomic(
    data,
    output_path,
):

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = (
        output_path.with_suffix(
            output_path.suffix
            + ".tmp"
        )
    )


    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            data,
            file,
            indent=2,
            default=make_json_serializable,
        )


    os.replace(
        temporary_path,
        output_path,
    )


# ------------------------------------------------------------
# 3. Atomic CSV writer
# ------------------------------------------------------------

def save_dataframe_atomic(
    dataframe,
    output_path,
):

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = (
        output_path.with_suffix(
            output_path.suffix
            + ".tmp"
        )
    )


    dataframe.to_csv(
        temporary_path,
        index=False,
    )


    os.replace(
        temporary_path,
        output_path,
    )


# ------------------------------------------------------------
# 4. Create the directory for one repeat / fold
# ------------------------------------------------------------

def get_resnet50_svm_fold_directory(
    repeat_number,
    fold_number,
):

    fold_directory = (
        REPEATED_RESNET50_SVM_DIR
        / f"repeat_{repeat_number:02d}"
        / f"fold_{fold_number:02d}"
    )


    return fold_directory


# ------------------------------------------------------------
# 5. Define the files required for a completed fold
# ------------------------------------------------------------

def get_resnet50_svm_fold_paths(
    repeat_number,
    fold_number,
):

    fold_directory = (
        get_resnet50_svm_fold_directory(
            repeat_number,
            fold_number,
        )
    )


    return {

        "directory":
            fold_directory,

        "search_results":
            (
                fold_directory
                / "inner_search_results.csv"
            ),

        "selected_parameters":
            (
                fold_directory
                / "selected_parameters.json"
            ),

        "outer_predictions":
            (
                fold_directory
                / "outer_predictions.csv"
            ),

        "outer_metrics":
            (
                fold_directory
                / "outer_metrics.json"
            ),

        "completion_marker":
            (
                fold_directory
                / "COMPLETED.json"
            ),
    }


# ------------------------------------------------------------
# 6. Determine whether one fold is already complete
#
# The completion marker is written LAST.
# Therefore its existence indicates that all required
# results were successfully saved.
# ------------------------------------------------------------

def is_resnet50_svm_fold_complete(
    repeat_number,
    fold_number,
):

    fold_paths = (
        get_resnet50_svm_fold_paths(
            repeat_number,
            fold_number,
        )
    )


    required_files = [

        fold_paths[
            "search_results"
        ],

        fold_paths[
            "selected_parameters"
        ],

        fold_paths[
            "outer_predictions"
        ],

        fold_paths[
            "outer_metrics"
        ],

        fold_paths[
            "completion_marker"
        ],
    ]


    return all(
        file_path.exists()
        for file_path
        in required_files
    )


# ------------------------------------------------------------
# 7. Validate directory structure using Repeat 1 / Fold 1
#
# This does NOT train anything.
# ------------------------------------------------------------

test_svm_fold_paths = (
    get_resnet50_svm_fold_paths(
        repeat_number=1,
        fold_number=1,
    )
)


assert (
    test_svm_fold_paths[
        "directory"
    ]
    ==
    (
        REPEATED_RESNET50_SVM_DIR
        / "repeat_01"
        / "fold_01"
    )
)


# ------------------------------------------------------------
# 8. Display
# ------------------------------------------------------------

print("=" * 70)

print(
    "RESNET50 FIXED-FEATURE + SVM "
    "PERSISTENCE / RESUME SETUP"
)

print("=" * 70)


print(
    "Results root:",
    REPEATED_RESNET50_SVM_DIR,
)


print(
    "\nExample fold directory:"
)

print(
    test_svm_fold_paths[
        "directory"
    ]
)


print(
    "\nFiles saved for every completed fold:"
)

print(
    " - inner_search_results.csv"
)

print(
    " - selected_parameters.json"
)

print(
    " - outer_predictions.csv"
)

print(
    " - outer_metrics.json"
)

print(
    " - COMPLETED.json"
)


print(
    "\nRepeat 1 / Fold 1 currently complete:",
    is_resnet50_svm_fold_complete(
        1,
        1,
    ),
)


print(
    "\nIMPORTANT:"
)

print(
    "The completion marker will be written only "
    "after all fold outputs are saved."
)

print(
    "After a Colab disconnect, completed folds "
    "can therefore be skipped safely."
)


print(
    "\nHybrid Step 9 SVM persistence / resume "
    "setup PASSED."
)

RESNET50 FIXED-FEATURE + SVM PERSISTENCE / RESUME SETUP
Results root: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50_svm

Example fold directory:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50_svm/repeat_01/fold_01

Files saved for every completed fold:
 - inner_search_results.csv
 - selected_parameters.json
 - outer_predictions.csv
 - outer_metrics.json
 - COMPLETED.json

Repeat 1 / Fold 1 currently complete: True

IMPORTANT:
The completion marker will be written only after all fold outputs are saved.
After a Colab disconnect, completed folds can therefore be skipped safely.

Hybrid Step 9 SVM persistence / resume setup PASSED.


In [10]:
# ============================================================
# Step 10. RUN AND VALIDATE ONE REPEATED NESTED-CV
# RESNET50 FIXED-FEATURE + SVM OUTER FOLD
#
# Test fold:
#   Repeat 1 / Fold 1
#
# This is a REAL experimental fold:
#   1. Obtain the shared outer partition.
#   2. Tune SVM hyperparameters using inner 3-fold CV
#      on the 4,480 outer-training samples only.
#   3. Refit the selected pipeline on all 4,480 samples.
#   4. Evaluate once on the untouched 1,120 outer-validation
#      samples.
#   5. Save all outputs to Google Drive.
#   6. Write the completion marker LAST.
#
# The full 10 × 5 runner will later skip this fold.
# ============================================================

import time

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


# ------------------------------------------------------------
# 1. Select the test repeat / fold
# ------------------------------------------------------------

TEST_REPEAT = 1
TEST_FOLD = 1


# ------------------------------------------------------------
# 2. Obtain the shared outer partition
# ------------------------------------------------------------

partition = (
    make_hybrid_outer_partition(
        repeat_number=TEST_REPEAT,
        fold_number=TEST_FOLD,
    )
)


X_outer_train = (
    partition[
        "outer_training_features"
    ]
)

y_outer_train = (
    partition[
        "outer_training_labels"
    ]
)

X_outer_validation = (
    partition[
        "outer_validation_features"
    ]
)

y_outer_validation = (
    partition[
        "outer_validation_labels"
    ]
)

outer_validation_paths = (
    partition[
        "outer_validation_paths"
    ]
)


assert X_outer_train.shape == (
    4480,
    FEATURE_DIMENSION,
)

assert X_outer_validation.shape == (
    1120,
    FEATURE_DIMENSION,
)


# ------------------------------------------------------------
# 3. Create a fresh fold-specific SVM search
# ------------------------------------------------------------

svm_search = (
    build_hybrid_svm_search(
        repeat_number=TEST_REPEAT,
        fold_number=TEST_FOLD,
    )
)


inner_cv_seed = (
    get_hybrid_inner_cv_seed(
        TEST_REPEAT,
        TEST_FOLD,
    )
)


# ------------------------------------------------------------
# 4. Run nested inner hyperparameter selection
#
# IMPORTANT:
# Only the 4,480 outer-training samples are supplied here.
# ------------------------------------------------------------

print("=" * 70)
print(
    "RUNNING RESNET50 FIXED-FEATURE + SVM"
)
print(
    "REPEAT 1 / FOLD 1"
)
print("=" * 70)

print(
    "Outer Training:",
    X_outer_train.shape,
)

print(
    "Outer validation:",
    X_outer_validation.shape,
)

print(
    "Inner-CV seed:",
    inner_cv_seed,
)

print(
    "Candidate configurations:",
    25,
)

print(
    "\nStarting inner 3-fold GridSearchCV..."
)


fold_start_time = (
    time.perf_counter()
)


svm_search.fit(
    X_outer_train,
    y_outer_train,
)


search_elapsed_seconds = (
    time.perf_counter()
    - fold_start_time
)


# ------------------------------------------------------------
# 5. Verify the search
#
# GridSearchCV(refit=True) has already refitted the winning
# pipeline on the complete 4,480-sample outer-training set.
# ------------------------------------------------------------

assert hasattr(
    svm_search,
    "best_estimator_",
)

assert hasattr(
    svm_search,
    "best_params_",
)

assert len(
    svm_search.cv_results_[
        "params"
    ]
) == 25


best_inner_macro_f1 = float(
    svm_search.best_score_
)


# ------------------------------------------------------------
# 6. Evaluate ONCE on the untouched outer-validation fold
# ------------------------------------------------------------

outer_predictions = (
    svm_search.predict(
        X_outer_validation
    )
)


assert len(
    outer_predictions
) == 1120


# ------------------------------------------------------------
# 7. Calculate outer-fold metrics
# ------------------------------------------------------------

outer_accuracy = float(
    accuracy_score(
        y_outer_validation,
        outer_predictions,
    )
)


outer_balanced_accuracy = float(
    balanced_accuracy_score(
        y_outer_validation,
        outer_predictions,
    )
)


outer_macro_f1 = float(
    f1_score(
        y_outer_validation,
        outer_predictions,
        average="macro",
    )
)


outer_macro_precision = float(
    precision_score(
        y_outer_validation,
        outer_predictions,
        average="macro",
        zero_division=0,
    )
)


outer_macro_recall = float(
    recall_score(
        y_outer_validation,
        outer_predictions,
        average="macro",
        zero_division=0,
    )
)


outer_confusion_matrix = (
    confusion_matrix(
        y_outer_validation,
        outer_predictions,
        labels=[
            0,
            1,
            2,
            3,
        ],
    )
)


outer_classification_report = (
    classification_report(
        y_outer_validation,
        outer_predictions,
        labels=[
            0,
            1,
            2,
            3,
        ],
        target_names=CLASS_NAMES,
        output_dict=True,
        zero_division=0,
    )
)


# ------------------------------------------------------------
# 8. Prepare inner-search results
# ------------------------------------------------------------

search_results_df = (
    pd.DataFrame(
        svm_search.cv_results_
    )
)


assert len(
    search_results_df
) == 25


# ------------------------------------------------------------
# 9. Prepare outer-validation predictions
# ------------------------------------------------------------

predictions_df = pd.DataFrame(
    {
        "relative_path":
            outer_validation_paths,

        "true_label":
            y_outer_validation,

        "predicted_label":
            outer_predictions,
    }
)


predictions_df[
    "true_class"
] = (
    predictions_df[
        "true_label"
    ]
    .map(
        INDEX_TO_CLASS
    )
)


predictions_df[
    "predicted_class"
] = (
    predictions_df[
        "predicted_label"
    ]
    .map(
        INDEX_TO_CLASS
    )
)


assert len(
    predictions_df
) == 1120


# ------------------------------------------------------------
# 10. Prepare selected-parameter record
# ------------------------------------------------------------

selected_parameters = {
    "model":
        "ResNet50_fixed_features_SVM",

    "repeat":
        TEST_REPEAT,

    "fold":
        TEST_FOLD,

    "outer_split_seed":
        partition[
            "split_seed"
        ],

    "inner_cv_seed":
        inner_cv_seed,

    "inner_cv_folds":
        NUMBER_OF_INNER_FOLDS,

    "selection_metric":
        "f1_macro",

    "number_of_candidates":
        25,

    "best_inner_macro_f1":
        best_inner_macro_f1,

    "best_parameters":
        svm_search.best_params_,
}


# ------------------------------------------------------------
# 11. Prepare outer-metric record
# ------------------------------------------------------------

outer_metrics = {
    "model":
        "ResNet50_fixed_features_SVM",

    "repeat":
        TEST_REPEAT,

    "fold":
        TEST_FOLD,

    "outer_split_seed":
        partition[
            "split_seed"
        ],

    "outer_training_samples":
        4480,

    "outer_validation_samples":
        1120,

    "best_inner_macro_f1":
        best_inner_macro_f1,

    "accuracy":
        outer_accuracy,

    "balanced_accuracy":
        outer_balanced_accuracy,

    "macro_f1":
        outer_macro_f1,

    "macro_precision":
        outer_macro_precision,

    "macro_recall":
        outer_macro_recall,

    "confusion_matrix":
        outer_confusion_matrix.tolist(),

    "classification_report":
        outer_classification_report,

    "elapsed_seconds":
        float(
            search_elapsed_seconds
        ),
}


# ------------------------------------------------------------
# 12. Obtain persistent fold paths
# ------------------------------------------------------------

fold_paths = (
    get_resnet50_svm_fold_paths(
        TEST_REPEAT,
        TEST_FOLD,
    )
)


fold_paths[
    "directory"
].mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# 13. Save every substantive output FIRST
# ------------------------------------------------------------

save_dataframe_atomic(
    search_results_df,
    fold_paths[
        "search_results"
    ],
)


save_json_atomic(
    selected_parameters,
    fold_paths[
        "selected_parameters"
    ],
)


save_dataframe_atomic(
    predictions_df,
    fold_paths[
        "outer_predictions"
    ],
)


save_json_atomic(
    outer_metrics,
    fold_paths[
        "outer_metrics"
    ],
)


# ------------------------------------------------------------
# 14. Read the saved files back and validate them
# ------------------------------------------------------------

saved_search_results = (
    pd.read_csv(
        fold_paths[
            "search_results"
        ]
    )
)


saved_predictions = (
    pd.read_csv(
        fold_paths[
            "outer_predictions"
        ]
    )
)


with open(
    fold_paths[
        "outer_metrics"
    ],
    "r",
    encoding="utf-8",
) as file:

    saved_outer_metrics = (
        json.load(
            file
        )
    )


assert len(
    saved_search_results
) == 25


assert len(
    saved_predictions
) == 1120


assert (
    saved_outer_metrics[
        "repeat"
    ]
    == TEST_REPEAT
)


assert (
    saved_outer_metrics[
        "fold"
    ]
    == TEST_FOLD
)


# ------------------------------------------------------------
# 15. Write the completion marker LAST
# ------------------------------------------------------------

completion_information = {
    "status":
        "completed",

    "model":
        "ResNet50_fixed_features_SVM",

    "repeat":
        TEST_REPEAT,

    "fold":
        TEST_FOLD,

    "required_outputs_verified":
        True,
}


save_json_atomic(
    completion_information,
    fold_paths[
        "completion_marker"
    ],
)


# ------------------------------------------------------------
# 16. Final completion verification
# ------------------------------------------------------------

assert (
    is_resnet50_svm_fold_complete(
        TEST_REPEAT,
        TEST_FOLD,
    )
), (
    "Repeat 1 / Fold 1 did not pass "
    "the completion check."
)


# ------------------------------------------------------------
# 17. Display result
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 70
)

print(
    "REPEAT 1 / FOLD 1 COMPLETED"
)

print(
    "=" * 70
)


print(
    "Best parameters:",
    svm_search.best_params_,
)


print(
    "Best inner macro F1:",
    round(
        best_inner_macro_f1,
        6,
    ),
)


print(
    "\nOuter accuracy:",
    round(
        outer_accuracy,
        6,
    ),
)


print(
    "Outer balanced accuracy:",
    round(
        outer_balanced_accuracy,
        6,
    ),
)


print(
    "Outer macro F1:",
    round(
        outer_macro_f1,
        6,
    ),
)


print(
    "Outer macro precision:",
    round(
        outer_macro_precision,
        6,
    ),
)


print(
    "Outer macro recall:",
    round(
        outer_macro_recall,
        6,
    ),
)


print(
    "\nElapsed seconds:",
    round(
        search_elapsed_seconds,
        2,
    ),
)


print(
    "\nSaved to:"
)

print(
    fold_paths[
        "directory"
    ]
)


print(
    "\nCompletion status:",
    is_resnet50_svm_fold_complete(
        TEST_REPEAT,
        TEST_FOLD,
    ),
)


print(
    "\nHybrid Step 10 SVM outer-fold "
    "validation PASSED."
)

RUNNING RESNET50 FIXED-FEATURE + SVM
REPEAT 1 / FOLD 1
Outer Training: (4480, 2048)
Outer validation: (1120, 2048)
Inner-CV seed: 303101
Candidate configurations: 25

Starting inner 3-fold GridSearchCV...


KeyboardInterrupt: 

In [20]:
# ============================================================
# Step 11. RUN ALL 10 × 5 REPEATED NESTED-CV
# RESNET50 FIXED-FEATURE + SVM FOLDS
#
# Total:
#   10 repeats × 5 outer folds = 50 outer evaluations
#
# Resume behaviour:
#   - completed + valid fold -> SKIP
#   - incomplete / corrupt fold -> DELETE and RERUN
#   - save immediately after every completed fold
#
# IMPORTANT:
#   - ResNet50 features are NOT extracted again.
#   - The existing 5,600 × 2,048 feature cache is reused.
#   - Hyperparameter selection occurs only inside the
#     4,480-sample outer-training partition.
#   - The 1,120 outer-validation samples are evaluated once.
#   - Testing data is never used.
# ============================================================

import gc
import json
import shutil
import time

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


# ------------------------------------------------------------
# 1. Validate an already-completed fold
#
# A fold is skipped only if:
#   - every required file exists
#   - completion marker says completed
#   - repeat/fold identifiers are correct
#   - 25 SVM search configurations were saved
#   - 1,120 outer predictions were saved
# ------------------------------------------------------------

def validate_completed_resnet50_svm_fold(
    repeat_number,
    fold_number,
):

    fold_paths = (
        get_resnet50_svm_fold_paths(
            repeat_number,
            fold_number,
        )
    )


    required_paths = [
        fold_paths["search_results"],
        fold_paths["selected_parameters"],
        fold_paths["outer_predictions"],
        fold_paths["outer_metrics"],
        fold_paths["completion_marker"],
    ]


    if not all(
        path.exists()
        for path in required_paths
    ):
        return False


    try:

        # ----------------------------------------------------
        # Read completion marker
        # ----------------------------------------------------

        with open(
            fold_paths[
                "completion_marker"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            completion = (
                json.load(
                    file
                )
            )


        if (
            completion.get(
                "status"
            )
            != "completed"
        ):
            return False


        if (
            int(
                completion.get(
                    "repeat"
                )
            )
            != repeat_number
        ):
            return False


        if (
            int(
                completion.get(
                    "fold"
                )
            )
            != fold_number
        ):
            return False


        # ----------------------------------------------------
        # Read metrics
        # ----------------------------------------------------

        with open(
            fold_paths[
                "outer_metrics"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            metrics = (
                json.load(
                    file
                )
            )


        if (
            int(
                metrics[
                    "repeat"
                ]
            )
            != repeat_number
        ):
            return False


        if (
            int(
                metrics[
                    "fold"
                ]
            )
            != fold_number
        ):
            return False


        if (
            int(
                metrics[
                    "outer_training_samples"
                ]
            )
            != 4480
        ):
            return False


        if (
            int(
                metrics[
                    "outer_validation_samples"
                ]
            )
            != 1120
        ):
            return False


        # ----------------------------------------------------
        # Read saved predictions
        # ----------------------------------------------------

        saved_predictions = (
            pd.read_csv(
                fold_paths[
                    "outer_predictions"
                ]
            )
        )


        if (
            len(
                saved_predictions
            )
            != 1120
        ):
            return False


        # ----------------------------------------------------
        # Read saved search results
        # ----------------------------------------------------

        saved_search_results = (
            pd.read_csv(
                fold_paths[
                    "search_results"
                ]
            )
        )


        if (
            len(
                saved_search_results
            )
            != 25
        ):
            return False


        # ----------------------------------------------------
        # Read selected parameters
        # ----------------------------------------------------

        with open(
            fold_paths[
                "selected_parameters"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            selected_parameters = (
                json.load(
                    file
                )
            )


        if (
            int(
                selected_parameters[
                    "repeat"
                ]
            )
            != repeat_number
        ):
            return False


        if (
            int(
                selected_parameters[
                    "fold"
                ]
            )
            != fold_number
        ):
            return False


        return True


    except Exception:

        return False


# ------------------------------------------------------------
# 2. Counters
# ------------------------------------------------------------

completed_folds = 0

skipped_folds = 0

rerun_folds = 0


full_run_start_time = (
    time.perf_counter()
)


# ------------------------------------------------------------
# 3. Run all 10 repeats × 5 outer folds
# ------------------------------------------------------------

for repeat_number in range(
    1,
    NUMBER_OF_REPEATS + 1,
):

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"REPEAT {repeat_number} / "
        f"{NUMBER_OF_REPEATS}"
    )

    print(
        "=" * 72
    )


    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    ):

        print(
            "\n"
            + "-" * 72
        )

        print(
            f"Repeat {repeat_number} / "
            f"Fold {fold_number}"
        )

        print(
            "-" * 72
        )


        # ----------------------------------------------------
        # Persistent fold paths
        # ----------------------------------------------------

        fold_paths = (
            get_resnet50_svm_fold_paths(
                repeat_number,
                fold_number,
            )
        )


        fold_directory = (
            fold_paths[
                "directory"
            ]
        )


        # ----------------------------------------------------
        # Skip already-completed valid folds
        # ----------------------------------------------------

        if (
            validate_completed_resnet50_svm_fold(
                repeat_number,
                fold_number,
            )
        ):

            print(
                "Status: already completed "
                "and validated."
            )

            print(
                "Action: SKIPPING."
            )


            skipped_folds += 1

            continue


        # ----------------------------------------------------
        # Delete an incomplete/corrupt previous attempt
        # ----------------------------------------------------

        if fold_directory.exists():

            print(
                "Existing fold directory is incomplete "
                "or invalid."
            )

            print(
                "Deleting incomplete fold before rerun..."
            )


            shutil.rmtree(
                fold_directory
            )


            rerun_folds += 1


        fold_directory.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ----------------------------------------------------
        # Obtain the correct shared outer partition
        # ----------------------------------------------------

        partition = (
            make_hybrid_outer_partition(
                repeat_number=
                    repeat_number,

                fold_number=
                    fold_number,
            )
        )


        X_outer_train = (
            partition[
                "outer_training_features"
            ]
        )


        y_outer_train = (
            partition[
                "outer_training_labels"
            ]
        )


        X_outer_validation = (
            partition[
                "outer_validation_features"
            ]
        )


        y_outer_validation = (
            partition[
                "outer_validation_labels"
            ]
        )


        outer_validation_paths = (
            partition[
                "outer_validation_paths"
            ]
        )


        assert X_outer_train.shape == (
            4480,
            FEATURE_DIMENSION,
        )


        assert X_outer_validation.shape == (
            1120,
            FEATURE_DIMENSION,
        )


        # ----------------------------------------------------
        # Fold-specific deterministic inner-CV seed
        # ----------------------------------------------------

        inner_cv_seed = (
            get_hybrid_inner_cv_seed(
                repeat_number,
                fold_number,
            )
        )


        # ----------------------------------------------------
        # Fresh SVM hyperparameter search
        # ----------------------------------------------------

        svm_search = (
            build_hybrid_svm_search(
                repeat_number,
                fold_number,
            )
        )


        print(
            "Split seed:",
            partition[
                "split_seed"
            ],
        )


        print(
            "Inner-CV seed:",
            inner_cv_seed,
        )


        print(
            "Outer Training:",
            X_outer_train.shape,
        )


        print(
            "Outer validation:",
            X_outer_validation.shape,
        )


        print(
            "SVM candidates:",
            25,
        )


        print(
            "\nStarting inner 3-fold "
            "hyperparameter search..."
        )


        fold_start_time = (
            time.perf_counter()
        )


        # ----------------------------------------------------
        # NESTED MODEL SELECTION
        #
        # Only outer-training data enters GridSearchCV.
        #
        # GridSearchCV(refit=True) automatically refits
        # the winning StandardScaler + SVM pipeline using
        # all 4,480 outer-training samples.
        # ----------------------------------------------------

        svm_search.fit(
            X_outer_train,
            y_outer_train,
        )


        search_elapsed_seconds = (
            time.perf_counter()
            - fold_start_time
        )


        # ----------------------------------------------------
        # Validate completed inner search
        # ----------------------------------------------------

        assert hasattr(
            svm_search,
            "best_estimator_",
        )


        assert hasattr(
            svm_search,
            "best_params_",
        )


        assert (
            len(
                svm_search.cv_results_[
                    "params"
                ]
            )
            == 25
        )


        best_inner_macro_f1 = float(
            svm_search.best_score_
        )


        # ----------------------------------------------------
        # OUTER EVALUATION
        #
        # Outer-validation data is used only here.
        # ----------------------------------------------------

        outer_predictions = (
            svm_search.predict(
                X_outer_validation
            )
        )


        assert (
            len(
                outer_predictions
            )
            == 1120
        )


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        outer_accuracy = float(
            accuracy_score(
                y_outer_validation,
                outer_predictions,
            )
        )


        outer_balanced_accuracy = float(
            balanced_accuracy_score(
                y_outer_validation,
                outer_predictions,
            )
        )


        outer_macro_f1 = float(
            f1_score(
                y_outer_validation,
                outer_predictions,
                average="macro",
            )
        )


        outer_macro_precision = float(
            precision_score(
                y_outer_validation,
                outer_predictions,
                average="macro",
                zero_division=0,
            )
        )


        outer_macro_recall = float(
            recall_score(
                y_outer_validation,
                outer_predictions,
                average="macro",
                zero_division=0,
            )
        )


        outer_confusion_matrix = (
            confusion_matrix(
                y_outer_validation,
                outer_predictions,
                labels=[
                    0,
                    1,
                    2,
                    3,
                ],
            )
        )


        outer_classification_report = (
            classification_report(
                y_outer_validation,
                outer_predictions,
                labels=[
                    0,
                    1,
                    2,
                    3,
                ],
                target_names=
                    CLASS_NAMES,
                output_dict=True,
                zero_division=0,
            )
        )


        # ----------------------------------------------------
        # Save complete inner-search results
        # ----------------------------------------------------

        search_results_df = (
            pd.DataFrame(
                svm_search.cv_results_
            )
        )


        assert (
            len(
                search_results_df
            )
            == 25
        )


        # ----------------------------------------------------
        # Outer-validation prediction table
        # ----------------------------------------------------

        predictions_df = pd.DataFrame(
            {
                "relative_path":
                    outer_validation_paths,

                "true_label":
                    y_outer_validation,

                "predicted_label":
                    outer_predictions,
            }
        )


        predictions_df[
            "true_class"
        ] = (
            predictions_df[
                "true_label"
            ]
            .map(
                INDEX_TO_CLASS
            )
        )


        predictions_df[
            "predicted_class"
        ] = (
            predictions_df[
                "predicted_label"
            ]
            .map(
                INDEX_TO_CLASS
            )
        )


        assert (
            len(
                predictions_df
            )
            == 1120
        )


        # ----------------------------------------------------
        # Selected hyperparameter record
        # ----------------------------------------------------

        selected_parameters = {

            "model":
                "ResNet50_fixed_features_SVM",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "outer_split_seed":
                int(
                    partition[
                        "split_seed"
                    ]
                ),

            "inner_cv_seed":
                int(
                    inner_cv_seed
                ),

            "inner_cv_folds":
                int(
                    NUMBER_OF_INNER_FOLDS
                ),

            "selection_metric":
                "f1_macro",

            "number_of_candidates":
                25,

            "best_inner_macro_f1":
                float(
                    best_inner_macro_f1
                ),

            "best_parameters":
                svm_search.best_params_,
        }


        # ----------------------------------------------------
        # Outer-fold metric record
        # ----------------------------------------------------

        outer_metrics = {

            "model":
                "ResNet50_fixed_features_SVM",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "outer_split_seed":
                int(
                    partition[
                        "split_seed"
                    ]
                ),

            "inner_cv_seed":
                int(
                    inner_cv_seed
                ),

            "outer_training_samples":
                4480,

            "outer_validation_samples":
                1120,

            "best_inner_macro_f1":
                float(
                    best_inner_macro_f1
                ),

            "accuracy":
                float(
                    outer_accuracy
                ),

            "balanced_accuracy":
                float(
                    outer_balanced_accuracy
                ),

            "macro_f1":
                float(
                    outer_macro_f1
                ),

            "macro_precision":
                float(
                    outer_macro_precision
                ),

            "macro_recall":
                float(
                    outer_macro_recall
                ),

            "confusion_matrix":
                outer_confusion_matrix.tolist(),

            "classification_report":
                outer_classification_report,

            "elapsed_seconds":
                float(
                    search_elapsed_seconds
                ),
        }


        # ----------------------------------------------------
        # Save substantive files FIRST
        # ----------------------------------------------------

        save_dataframe_atomic(
            search_results_df,
            fold_paths[
                "search_results"
            ],
        )


        save_json_atomic(
            selected_parameters,
            fold_paths[
                "selected_parameters"
            ],
        )


        save_dataframe_atomic(
            predictions_df,
            fold_paths[
                "outer_predictions"
            ],
        )


        save_json_atomic(
            outer_metrics,
            fold_paths[
                "outer_metrics"
            ],
        )


        # ----------------------------------------------------
        # Read saved outputs back before declaring completion
        # ----------------------------------------------------

        saved_search_results = (
            pd.read_csv(
                fold_paths[
                    "search_results"
                ]
            )
        )


        saved_predictions = (
            pd.read_csv(
                fold_paths[
                    "outer_predictions"
                ]
            )
        )


        with open(
            fold_paths[
                "outer_metrics"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            saved_metrics = (
                json.load(
                    file
                )
            )


        assert (
            len(
                saved_search_results
            )
            == 25
        )


        assert (
            len(
                saved_predictions
            )
            == 1120
        )


        assert (
            int(
                saved_metrics[
                    "repeat"
                ]
            )
            == repeat_number
        )


        assert (
            int(
                saved_metrics[
                    "fold"
                ]
            )
            == fold_number
        )


        # ----------------------------------------------------
        # Completion marker MUST be written LAST
        # ----------------------------------------------------

        completion_information = {

            "status":
                "completed",

            "model":
                "ResNet50_fixed_features_SVM",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "required_outputs_verified":
                True,
        }


        save_json_atomic(
            completion_information,
            fold_paths[
                "completion_marker"
            ],
        )


        # ----------------------------------------------------
        # Final saved-fold validation
        # ----------------------------------------------------

        assert (
            validate_completed_resnet50_svm_fold(
                repeat_number,
                fold_number,
            )
        ), (
            "Saved fold failed final "
            "completion validation."
        )


        completed_folds += 1


        # ----------------------------------------------------
        # Fold status
        # ----------------------------------------------------

        print(
            "\nBest parameters:",
            svm_search.best_params_,
        )


        print(
            "Best inner macro F1:",
            round(
                best_inner_macro_f1,
                6,
            ),
        )


        print(
            "Outer accuracy:",
            round(
                outer_accuracy,
                6,
            ),
        )


        print(
            "Outer balanced accuracy:",
            round(
                outer_balanced_accuracy,
                6,
            ),
        )


        print(
            "Outer macro F1:",
            round(
                outer_macro_f1,
                6,
            ),
        )


        print(
            "Elapsed seconds:",
            round(
                search_elapsed_seconds,
                2,
            ),
        )


        print(
            "Status: SAVED AND VERIFIED."
        )


        # ----------------------------------------------------
        # Release fold-specific objects
        # ----------------------------------------------------

        del svm_search
        del partition

        del X_outer_train
        del y_outer_train

        del X_outer_validation
        del y_outer_validation

        del outer_predictions

        gc.collect()


    # --------------------------------------------------------
    # Repeat completion status
    # --------------------------------------------------------

    completed_in_repeat = sum(

        validate_completed_resnet50_svm_fold(
            repeat_number,
            fold_number,
        )

        for fold_number
        in range(
            1,
            NUMBER_OF_OUTER_FOLDS + 1,
        )
    )


    print(
        "\n"
        f"Repeat {repeat_number} status: "
        f"{completed_in_repeat}/5 folds complete."
    )


# ------------------------------------------------------------
# 4. Final experiment validation
# ------------------------------------------------------------

total_valid_completed_folds = sum(

    validate_completed_resnet50_svm_fold(
        repeat_number,
        fold_number,
    )

    for repeat_number
    in range(
        1,
        NUMBER_OF_REPEATS + 1,
    )

    for fold_number
    in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    )
)


full_run_elapsed_seconds = (
    time.perf_counter()
    - full_run_start_time
)


# ------------------------------------------------------------
# 5. Final status
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "RESNET50 FIXED-FEATURE + SVM "
    "REPEATED NESTED-CV RUN FINISHED"
)

print(
    "=" * 72
)


print(
    "Valid completed folds:",
    total_valid_completed_folds,
    "/ 50",
)


print(
    "New folds completed this run:",
    completed_folds,
)


print(
    "Previously completed folds skipped:",
    skipped_folds,
)


print(
    "Incomplete/corrupt folds rerun:",
    rerun_folds,
)


print(
    "Total runtime this session (seconds):",
    round(
        full_run_elapsed_seconds,
        2,
    ),
)


print(
    "\nResults root:"
)

print(
    REPEATED_RESNET50_SVM_DIR
)


if (
    total_valid_completed_folds
    == 50
):

    print(
        "\nALL 50 RESNET50 + SVM "
        "OUTER FOLDS COMPLETED AND VERIFIED."
    )

else:

    print(
        "\nExperiment is not yet complete."
    )

    print(
        "Rerun this same cell after reconnecting."
    )

    print(
        "Valid completed folds will be skipped."
    )


print(
    "\nTesting partition was not used."
)


REPEAT 1 / 10

------------------------------------------------------------------------
Repeat 1 / Fold 1
------------------------------------------------------------------------
Status: already completed and validated.
Action: SKIPPING.

------------------------------------------------------------------------
Repeat 1 / Fold 2
------------------------------------------------------------------------
Status: already completed and validated.
Action: SKIPPING.

------------------------------------------------------------------------
Repeat 1 / Fold 3
------------------------------------------------------------------------
Status: already completed and validated.
Action: SKIPPING.

------------------------------------------------------------------------
Repeat 1 / Fold 4
------------------------------------------------------------------------
Status: already completed and validated.
Action: SKIPPING.

------------------------------------------------------------------------
Repeat 1 / Fold

In [21]:
# ============================================================
# ACCESS AND AGGREGATE RESNET50 FIXED-FEATURE + SVM RESULTS
#
# Reads the exact files produced by the repeated Hybrid 1
# SVM experiment.
#
# Expected structure:
#
# repeated_nested_cv/resnet50_svm/
#     repeat_01/
#         fold_01/
#             inner_search_results.csv
#             selected_parameters.json
#             outer_predictions.csv
#             outer_metrics.json
#             COMPLETED.json
#
# Produces:
#     all_fold_results.csv
#     repeat_level_summary.csv
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Results root
# ------------------------------------------------------------

RESULTS_ROOT = Path(
    "/content/drive/MyDrive/brain_tumour_colab/"
    "results/repeated_nested_cv/resnet50_svm"
)


assert RESULTS_ROOT.exists(), (
    "Results directory does not exist:\n"
    f"{RESULTS_ROOT}"
)


# ------------------------------------------------------------
# 2. Required files for every completed fold
# ------------------------------------------------------------

REQUIRED_FILES = [
    "inner_search_results.csv",
    "selected_parameters.json",
    "outer_predictions.csv",
    "outer_metrics.json",
    "COMPLETED.json",
]


# ------------------------------------------------------------
# 3. Scan all 50 fold directories
# ------------------------------------------------------------

fold_records = []

problem_folds = []


for repeat_number in range(1, 11):

    for fold_number in range(1, 6):

        fold_directory = (
            RESULTS_ROOT
            / f"repeat_{repeat_number:02d}"
            / f"fold_{fold_number:02d}"
        )


        # ----------------------------------------------------
        # Check folder
        # ----------------------------------------------------

        if not fold_directory.exists():

            problem_folds.append(
                {
                    "repeat": repeat_number,
                    "fold": fold_number,
                    "problem": "folder missing",
                    "path": str(fold_directory),
                }
            )

            continue


        # ----------------------------------------------------
        # Check required files
        # ----------------------------------------------------

        missing_files = [
            filename
            for filename
            in REQUIRED_FILES
            if not (
                fold_directory
                / filename
            ).exists()
        ]


        if missing_files:

            problem_folds.append(
                {
                    "repeat": repeat_number,
                    "fold": fold_number,
                    "problem":
                        "missing required files: "
                        + ", ".join(missing_files),
                    "path": str(fold_directory),
                }
            )

            continue


        # ----------------------------------------------------
        # Load completion marker
        # ----------------------------------------------------

        with open(
            fold_directory
            / "COMPLETED.json",
            "r",
            encoding="utf-8",
        ) as file:

            completion = json.load(
                file
            )


        if completion.get(
            "status"
        ) != "completed":

            problem_folds.append(
                {
                    "repeat": repeat_number,
                    "fold": fold_number,
                    "problem":
                        "completion marker does not "
                        "say completed",
                    "path": str(fold_directory),
                }
            )

            continue


        # ----------------------------------------------------
        # Load outer metrics
        # ----------------------------------------------------

        with open(
            fold_directory
            / "outer_metrics.json",
            "r",
            encoding="utf-8",
        ) as file:

            metrics = json.load(
                file
            )


        # ----------------------------------------------------
        # Load selected parameters
        # ----------------------------------------------------

        with open(
            fold_directory
            / "selected_parameters.json",
            "r",
            encoding="utf-8",
        ) as file:

            selected = json.load(
                file
            )


        best_parameters = (
            selected.get(
                "best_parameters",
                {},
            )
        )


        # ----------------------------------------------------
        # Build one fold-level record
        # ----------------------------------------------------

        fold_records.append(
            {
                "model":
                    metrics.get(
                        "model"
                    ),

                "repeat":
                    int(
                        metrics[
                            "repeat"
                        ]
                    ),

                "fold":
                    int(
                        metrics[
                            "fold"
                        ]
                    ),

                "outer_split_seed":
                    metrics.get(
                        "outer_split_seed"
                    ),

                "inner_cv_seed":
                    selected.get(
                        "inner_cv_seed"
                    ),

                "outer_training_samples":
                    metrics.get(
                        "outer_training_samples"
                    ),

                "outer_validation_samples":
                    metrics.get(
                        "outer_validation_samples"
                    ),

                "best_C":
                    best_parameters.get(
                        "classifier__C"
                    ),

                "best_gamma":
                    best_parameters.get(
                        "classifier__gamma"
                    ),

                "best_inner_macro_f1":
                    metrics.get(
                        "best_inner_macro_f1"
                    ),

                "accuracy":
                    metrics.get(
                        "accuracy"
                    ),

                "balanced_accuracy":
                    metrics.get(
                        "balanced_accuracy"
                    ),

                "macro_precision":
                    metrics.get(
                        "macro_precision"
                    ),

                "macro_recall":
                    metrics.get(
                        "macro_recall"
                    ),

                "macro_f1":
                    metrics.get(
                        "macro_f1"
                    ),

                "elapsed_seconds":
                    metrics.get(
                        "elapsed_seconds"
                    ),

                "fold_directory":
                    str(
                        fold_directory
                    ),
            }
        )


# ------------------------------------------------------------
# 4. Create fold-level table
# ------------------------------------------------------------

fold_results_df = pd.DataFrame(
    fold_records
)


if not fold_results_df.empty:

    fold_results_df = (
        fold_results_df
        .sort_values(
            [
                "repeat",
                "fold",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 5. Display scan result
# ------------------------------------------------------------

print("=" * 72)

print(
    "RESNET50 + SVM SAVED RESULT CHECK"
)

print("=" * 72)


print(
    "Results root:",
    RESULTS_ROOT,
)


print(
    "\nValid fold results found:",
    len(
        fold_results_df
    ),
    "/ 50",
)


print(
    "Problem folds:",
    len(
        problem_folds
    ),
)


if problem_folds:

    problem_folds_df = (
        pd.DataFrame(
            problem_folds
        )
    )

    print(
        "\nPROBLEM FOLDS:"
    )

    display(
        problem_folds_df
    )

else:

    print(
        "\nAll 50 fold directories contain "
        "the required saved files."
    )


# ------------------------------------------------------------
# 6. Stop aggregation if fewer than 50 valid folds exist
# ------------------------------------------------------------

if len(
    fold_results_df
) != 50:

    raise RuntimeError(
        "\nOnly "
        f"{len(fold_results_df)} / 50 "
        "valid fold result sets were found.\n"
        "See the problem-fold table above."
    )


# ------------------------------------------------------------
# 7. Validate repeated structure
# ------------------------------------------------------------

assert (
    fold_results_df[
        "repeat"
    ].nunique()
    == 10
)


assert (
    fold_results_df
    .groupby(
        "repeat"
    )
    .size()
    .eq(5)
    .all()
)


# ------------------------------------------------------------
# 8. Build the 10 repeat-level observations
#
# These mean macro-F1 values are the values later used
# for paired Wilcoxon comparison.
# ------------------------------------------------------------

repeat_level_summary_df = (
    fold_results_df
    .groupby(
        "repeat",
        as_index=False,
    )
    .agg(
        mean_accuracy=(
            "accuracy",
            "mean",
        ),

        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        mean_macro_precision=(
            "macro_precision",
            "mean",
        ),

        mean_macro_recall=(
            "macro_recall",
            "mean",
        ),

        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),

        sd_macro_f1=(
            "macro_f1",
            "std",
        ),
    )
)


repeat_level_summary_df[
    "wilcoxon_value"
] = (
    repeat_level_summary_df[
        "mean_macro_f1"
    ]
)


# ------------------------------------------------------------
# 9. Save consolidated tables
# ------------------------------------------------------------

ALL_FOLD_RESULTS_PATH = (
    RESULTS_ROOT
    / "all_fold_results.csv"
)


REPEAT_LEVEL_SUMMARY_PATH = (
    RESULTS_ROOT
    / "repeat_level_summary.csv"
)


fold_results_df.to_csv(
    ALL_FOLD_RESULTS_PATH,
    index=False,
)


repeat_level_summary_df.to_csv(
    REPEAT_LEVEL_SUMMARY_PATH,
    index=False,
)


# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "50 OUTER-FOLD RESULTS"
)

print(
    "=" * 72
)


display(
    fold_results_df
)


print(
    "\n"
    + "=" * 72
)

print(
    "10 REPEAT-LEVEL RESULTS"
)

print(
    "=" * 72
)


display(
    repeat_level_summary_df
)


print(
    "\nOverall mean outer-fold macro F1:",
    round(
        fold_results_df[
            "macro_f1"
        ].mean(),
        6,
    ),
)


print(
    "Mean of 10 repeat-level macro F1 values:",
    round(
        repeat_level_summary_df[
            "mean_macro_f1"
        ].mean(),
        6,
    ),
)


print(
    "\nSaved fold-level table:"
)

print(
    ALL_FOLD_RESULTS_PATH
)


print(
    "\nSaved repeat-level table:"
)

print(
    REPEAT_LEVEL_SUMMARY_PATH
)


print(
    "\nRESNET50 + SVM RESULT ACCESS PASSED."
)

RESNET50 + SVM SAVED RESULT CHECK
Results root: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50_svm

Valid fold results found: 50 / 50
Problem folds: 0

All 50 fold directories contain the required saved files.

50 OUTER-FOLD RESULTS


,model,repeat,fold,outer_split_seed,inner_cv_seed,outer_training_samples,outer_validation_samples,best_C,best_gamma,best_inner_macro_f1,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,elapsed_seconds,fold_directory
0,ResNet50_fixed_features_SVM,1,1,202601,303101,4480,1120,10,scale,0.940540,0.963393,0.963393,0.964282,0.963393,0.963537,1530.667471,/content/drive/MyDrive/brain_tumour_colab/resu...
1,ResNet50_fixed_features_SVM,1,2,202601,303102,4480,1120,10,scale,0.946815,0.954464,0.954464,0.955543,0.954464,0.954558,1489.084888,/content/drive/MyDrive/brain_tumour_colab/resu...
2,ResNet50_fixed_features_SVM,1,3,202601,303103,4480,1120,100,0.0001,0.940175,0.955357,0.955357,0.955732,0.955357,0.955462,1476.240939,/content/drive/MyDrive/brain_tumour_colab/resu...
3,ResNet50_fixed_features_SVM,1,4,202601,303104,4480,1120,10,scale,0.941420,0.959821,0.959821,0.960617,0.959821,0.959857,1477.913745,/content/drive/MyDrive/brain_tumour_colab/resu...
4,ResNet50_fixed_features_SVM,1,5,202601,303105,4480,1120,10,scale,0.942161,0.947321,0.947321,0.950970,0.947321,0.947683,1481.021127,/content/drive/MyDrive/brain_tumour_colab/resu...
5,ResNet50_fixed_features_SVM,2,1,202602,303201,4480,1120,100,0.0001,0.940588,0.951786,0.951786,0.952278,0.951786,0.951854,1471.170585,/content/drive/MyDrive/brain_tumour_colab/resu...
6,ResNet50_fixed_features_SVM,2,2,202602,303202,4480,1120,10,scale,0.940595,0.950000,0.950000,0.952832,0.950000,0.950456,1484.603794,/content/drive/MyDrive/brain_tumour_colab/resu...
7,ResNet50_fixed_features_SVM,2,3,202602,303203,4480,1120,10,scale,0.938501,0.969643,0.969643,0.969960,0.969643,0.969713,1474.245519,/content/drive/MyDrive/brain_tumour_colab/resu...
8,ResNet50_fixed_features_SVM,2,4,202602,303204,4480,1120,10,scale,0.942178,0.963393,0.963393,0.963960,0.963393,0.963474,1507.633168,/content/drive/MyDrive/brain_tumour_colab/resu...
9,ResNet50_fixed_features_SVM,2,5,202602,303205,4480,1120,10,scale,0.942946,0.951786,0.951786,0.954178,0.951786,0.952059,1472.166893,/content/drive/MyDrive/brain_tumour_colab/resu...



10 REPEAT-LEVEL RESULTS


,repeat,mean_accuracy,mean_balanced_accuracy,mean_macro_precision,mean_macro_recall,mean_macro_f1,sd_macro_f1,wilcoxon_value
0,1,0.956071,0.956071,0.957429,0.956071,0.956219,0.005979,0.956219
1,2,0.957321,0.957321,0.958642,0.957321,0.957511,0.008602,0.957511
2,3,0.958571,0.958571,0.959647,0.958571,0.958669,0.004768,0.958669
3,4,0.956786,0.956786,0.958023,0.956786,0.956985,0.010193,0.956985
4,5,0.957321,0.957321,0.958113,0.957321,0.957465,0.003917,0.957465
5,6,0.957857,0.957857,0.958759,0.957857,0.957992,0.007001,0.957992
6,7,0.953929,0.953929,0.955212,0.953929,0.954104,0.009167,0.954104
7,8,0.957679,0.957679,0.958647,0.957679,0.957819,0.003231,0.957819
8,9,0.955357,0.955357,0.956055,0.955357,0.955447,0.007680,0.955447
9,10,0.957500,0.957500,0.958687,0.957500,0.957660,0.005088,0.957660



Overall mean outer-fold macro F1: 0.956987
Mean of 10 repeat-level macro F1 values: 0.956987

Saved fold-level table:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50_svm/all_fold_results.csv

Saved repeat-level table:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/resnet50_svm/repeat_level_summary.csv

RESNET50 + SVM RESULT ACCESS PASSED.


In [ ]:
# ============================================================
# HYBRID EXPERIMENT SETUP AND REPRODUCIBILITY
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import time


# ============================================================
# Reproducibility
# ============================================================


# Set the seed for random number
RANDOM_SEED = 42

# Making experiment runs reproducible
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Making determenistic TesnsorFlow operations where supported
try:
  tf.config.experimental.enable_op_determinism()
  determinism_status = "enabled"

except Exception as error:
  determinism_status = ("requested but TensorFlow returned:" f" {error}")

# Keep TensorFlow/Keras calculations in float 32
tf.keras.backend.set_floatx("float32")


# ============================================================
# 2. Fixed experiment configuration
# ============================================================

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for index, class_name in enumerate(CLASS_NAMES)
}


# ============================================================
# 3. Verify experiment configuration
# ============================================================

print("=" * 70)
print("HYBRID EXPERIMENT SETUP")
print("=" * 70)
print("Random seed:", RANDOM_SEED)
print("Deterministic TensorFlow operations:", determinism_status)
print("Keras float type:", tf.keras.backend.floatx())
print("Input shape:", (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))
print("Number of classes:", NUMBER_OF_CLASSES)
print("Class mapping:", CLASS_TO_INDEX)





HYBRID EXPERIMENT SETUP
Random seed: 42
Deterministic TensorFlow operations: enabled
Keras float type: float32
Input shape: (224, 224, 3)
Number of classes: 4
Class mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [ ]:
# ============================================================
# LOAD AND VALIDATE THE FIXED FIVE-FOLD ASSIGNMENTS
# ============================================================

import pandas as pd

# Load the fixed five-fold Training assignments
assignments = pd.read_csv(FOLDS_FILE)


# ============================================================
# 1. Validate the table structure
# ============================================================

# Define the columns that must exist in the fold-assignment file
required_columns = {"relative_path", "class", "fold"}


# Find any required columns that are missing
missing_columns = (required_columns - set(assignments.columns))


# Stop if any required column is missing
if missing_columns:
    raise ValueError("The fold file is missing columns: "f"{sorted(missing_columns)}")



# ============================================================
# 2. Create class indices and complete image paths
# ============================================================

# Assign the fixed numeric class index to every Training image
assignments["class_index"] = (assignments["class"].map(CLASS_TO_INDEX))


# Create the complete path to every Training image
assignments["image_path"] = (assignments["relative_path"].apply(
        lambda path:
        DATA_DIR
        / Path(path)))


# ============================================================
# 3. Validate paths and assignments
# ============================================================

# Find any Training images that do not exist
missing_image_paths = [
    image_path
    for image_path
    in assignments["image_path"]
    if not image_path.exists()
]


# Find any class names that do not belong to the four expected classes
unexpected_classes = sorted(set(assignments["class"]) - set(CLASS_NAMES))


# Find any fold values outside the fixed folds 1 to 5
unexpected_folds = sorted(set(assignments["fold"]) - {1, 2, 3, 4, 5})


# Count duplicate Training image paths
duplicate_paths = (assignments["relative_path"].duplicated().sum())


# Count missing values in the required columns
missing_values = (
    assignments[["relative_path", "class", "fold"]].isna().sum())


# Check whether any class failed to receive a numeric class index
missing_class_indices = (assignments["class_index"].isna().sum())


# ============================================================
# 4. Verify the fixed fold distribution
# ============================================================

# Count how many images from every class belong to each outer fold
fold_distribution = pd.crosstab(assignments["fold"], assignments["class"])


# Put the class columns into the fixed project class order
fold_distribution = (fold_distribution.reindex(
        columns=CLASS_NAMES,
        fill_value=0))


# Add the total number of images in each fold
fold_distribution["total"] = (fold_distribution.sum(axis=1))


# Each of the five fixed folds must contain 280 images from every class
expected_fold_distribution = pd.DataFrame(
    {
        class_name: [280] * 5
        for class_name in CLASS_NAMES
    },
    index=[1, 2, 3, 4, 5])


expected_fold_distribution["total"] = 1120


# ============================================================
# 5. Display the validation results
# ============================================================

print("=" * 70)
print("FIXED FIVE-FOLD ASSIGNMENT VALIDATION")
print("=" * 70)
print("Rows:", len(assignments))
print("Columns:", assignments.columns.tolist())
print("\n--- Assignment checks ---")
print("Duplicate relative paths:", duplicate_paths)
print("Missing image files:", len(missing_image_paths))
print("Unexpected classes:", unexpected_classes)
print("Unexpected fold values:", unexpected_folds)
print("Missing class indices:", missing_class_indices)
print("\nMissing values:")
print(missing_values)
print("\n--- Images per fold and class ---")
display(fold_distribution)
print("\n--- Sample records ---")
display(
    assignments[["relative_path", "class", "class_index", "fold", "image_path"]].head())


# ============================================================
# 6. Stop if any validation check failed
# ============================================================

# The fold file must contain exactly 5,600 Training images
if len(assignments) != 5600:
    raise ValueError("Expected 5,600 Training images, "f"but found {len(assignments)}.")


# Every Training image path must be unique
if duplicate_paths != 0:
    raise ValueError("Duplicate paths were found in the fold file.")


# Every Training image listed in the CSV must exist
if missing_image_paths:
    raise FileNotFoundError(f"{len(missing_image_paths)} Training image files are missing.")


# Only the expected four classes are allowed
if unexpected_classes:
    raise ValueError(f"Unexpected classes found: {unexpected_classes}")


# Only folds 1 to 5 are allowed
if unexpected_folds:
    raise ValueError(f"Unexpected fold values found: {unexpected_folds}")


# Required columns must not contain missing values
if missing_values.sum() != 0:
    raise ValueError("Missing values were found in the fold file.")


# Every class must map successfully to its fixed numeric index
if missing_class_indices != 0:
    raise ValueError(
        "One or more Training classes could not be mapped to a class index.")


# Verify the exact fixed five-fold class distribution
if not fold_distribution.equals(expected_fold_distribution):
    raise ValueError(
        "The fixed five-fold class distribution does not match the expected "
        "280 images per class and 1,120 images per fold.")

print("\nFixed five-fold assignment validation passed.")

FIXED FIVE-FOLD ASSIGNMENT VALIDATION
Rows: 5600
Columns: ['relative_path', 'class', 'fold', 'class_index', 'image_path']

--- Assignment checks ---
Duplicate relative paths: 0
Missing image files: 0
Unexpected classes: []
Unexpected fold values: []
Missing class indices: 0

Missing values:
relative_path    0
class            0
fold             0
dtype: int64

--- Images per fold and class ---


class,glioma,meningioma,notumor,pituitary,total
fold,,,,,
1,280,280,280,280,1120
2,280,280,280,280,1120
3,280,280,280,280,1120
4,280,280,280,280,1120
5,280,280,280,280,1120



--- Sample records ---


,relative_path,class,class_index,fold,image_path
0,Training/glioma/Tr-gl_100.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
1,Training/glioma/Tr-gl_1001.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
2,Training/glioma/Tr-gl_1003.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
3,Training/glioma/Tr-gl_1014.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
4,Training/glioma/Tr-gl_1015.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...



Fixed five-fold assignment validation passed.


In [ ]:
# ============================================================
# CREATING THE FIXED IMAGENET RESNET50 FEATURE EXTRACTOR
# ============================================================


# Creating a fresh ImageNet-pretrained ResNet50 model to extract features from MRI images
resnet50_feature_extractor = tf.keras.applications.ResNet50(
    include_top=False,     # Remove ResNet50's original ImageNet classification head
    weights="imagenet",    # The ImageNet weights are used
    input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS),    # Dimentions of the image
    pooling="avg",    # Convert the final feature maps into 2,048 average feature values
)

# Freeze every ResNet50 layer
resnet50_feature_extractor.trainable = False

# Expected number of features produced for each MRI image
FEATURE_DIMENSION = 2048


# ============================================================
# VERIFY THE FEATURE EXTRACTOR
# ============================================================

print("=" * 70)
print("FIXED IMAGENET RESNET50 FEATURE EXTRACTOR")
print("=" * 70)

print("Model name:", resnet50_feature_extractor.name)
print("Input shape:", resnet50_feature_extractor.input_shape)
print("Output shape:", resnet50_feature_extractor.output_shape)
print("Trainable:", resnet50_feature_extractor.trainable)
print("Trainable parameters:",
    sum(np.prod(variable.shape) for variable in resnet50_feature_extractor.trainable_weights))


# The feature extractor must produce exactly 2,048 features
if resnet50_feature_extractor.output_shape[-1] != FEATURE_DIMENSION:
    raise ValueError(
        f"Expected {FEATURE_DIMENSION} features, "
        f"but got {resnet50_feature_extractor.output_shape[-1]}.")


# Checking for trainable weights
if resnet50_feature_extractor.trainable_weights:
  raise RuntimeError("The feature extractor must not have any trainable weights.")


print("\nFixed feature extractor validation passed.")




94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 5s 0us/step
FIXED IMAGENET RESNET50 FEATURE EXTRACTOR
Model name: resnet50
Input shape: (None, 224, 224, 3)
Output shape: (None, 2048)
Trainable: False
Trainable parameters: 0

Fixed feature extractor validation passed.


In [ ]:
# ============================================================
# DEFINING RESNET50 IMAGE PREPROCESSING FOR FEATURE EXTRACTION
# ============================================================

def load_and_preprocess_resnet50_image(image_path, class_index):
  """
  Load one grayscale MRI image, convert it to pseudo-RGB and apply the ImageNet50 preprocessing function
  """

  # Reading the PNG image file
  image_bytes = tf.io.read_file(image_path)

  # Decoding as a single-channel grayscale image
  grayscale_image = tf.io.decode_png(image_bytes, channels = 1)

  # Confirming the expected grayscale image shape
  grayscale_image = tf.ensure_shape(grayscale_image, (IMAGE_HEIGHT, IMAGE_WIDTH, 1))

  # Repeat the grayscale channel three times
  pseudo_rgb_image = tf.image.grayscale_to_rgb(grayscale_image)

  # Convert pixel values to float32
  pseudo_rgb_image = tf.cast(pseudo_rgb_image, tf.float32)

  # Apply the ImageNet50 preprocessing required by ImageNet ResNet50
  preprocessed_image = tf.keras.applications.resnet50.preprocess_input(pseudo_rgb_image)

  # Confirm the final ResNet50 input shape
  resnet50_image = tf.ensure_shape(preprocessed_image, (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))

  return resnet50_image, class_index


In [ ]:
# ============================================================
# CREATE THE RESNET50 FEATURE-EXTRACTION DATASET
# ============================================================

def create_resnet50_feature_dataset(image_paths, class_indices, batch_size):
    """
    Create a deterministic TensorFlow dataset for fixed ResNet50 feature extraction without shuffling.
    """

    # Creating the dataset from image paths and class indices
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, class_indices))

    # Keeping dataset processing deterministic
    dataset_options = tf.data.Options()
    dataset_options.experimental_deterministic = True

    dataset = dataset.with_options(dataset_options)

    # Loading and preprocessing each MRI image
    dataset = dataset.map(load_and_preprocess_resnet50_image, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)

    # Grouping images into batches without dropping the final batch
    dataset = dataset.batch(batch_size, drop_remainder=False)

    # Preparing upcoming batches in advance
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
# ============================================================
# DEFINE RESNET50 FEATURE EXTRACTION
# ============================================================


def extract_resnet50_features(dataset, feature_extractor):
  """
  Extract fixed ResNet50 features and preserve the corresponding class indices in dataset order.
  """

  feature_batches = []
  class_index_batches = []

  # Process the dataset one batch at a time
  for resnet50_image_batch, class_index_batch in dataset:

    # Extract fixed ImageNet ResNet50 features
    feature_batch = feature_extractor(resnet50_image_batch, training=False)

    # Store the features and corresponding class indices
    feature_batches.append(feature_batch.numpy())
    class_index_batches.append(class_index_batch.numpy())


  # Combining all batches into complete NumPy arrays
  features = np.concatenate(feature_batches, axis = 0).astype(np.float32)

  class_indices = np.concatenate(class_index_batches, axis = 0).astype(np.int32)

  return features, class_indices


In [ ]:
from numpy._core.multiarray import dtype

# ============================================================
# PREPARE THE TRAINING DATA FOR FEATURE EXTRACTION
# ============================================================

# Storing the Training image paths in fold-assignment row order
training_image_paths = assignments["image_path"].astype(str).to_numpy()


# Storing the Training class indices in fold-assignment row order
training_class_indices = assignments["class_index"].to_numpy(dtype = np.int32)


# Storing the corresponding fixed outer-fold assignments
training_fold_assignments = assignments["fold"].to_numpy(dtype = np.int32)


# Storing the relative paths for later feature-cache verification
training_relative_paths = assignments["relative_path"].astype(str).to_numpy()


# Verifying that all Training arrays contain the expected 5,600 rows
if not(len(training_image_paths) == len(training_class_indices) == len(training_fold_assignments)
    == len(training_relative_paths) == 5600):

  raise RuntimeError("The prepared Training arrays do not all the expected 5,600 images")


print("Training images prepared.", len(training_image_paths) )


Training images prepared. 5600


In [ ]:
# ============================================================
# CREATING THE TRAINING FEATURE EXTRACTION DATASET
# ============================================================

FEATURE_EXTRACTION_BATCH_SIZE = 64

training_feature_dataset = create_resnet50_feature_dataset(
    image_paths=training_image_paths,
    class_indices=training_class_indices,
    batch_size=FEATURE_EXTRACTION_BATCH_SIZE)

print("Training feature dataset created.")

Training feature dataset created.


In [ ]:
# ============================================================
# EXTRACTING RESNET50 FEATURES FROM THE TRAINING PARTITION
# ============================================================

training_feature_extraction_start = time.perf_counter()

# features and the class number
training_features, extracted_training_class_indices = extract_resnet50_features(
    dataset = training_feature_dataset, feature_extractor = resnet50_feature_extractor)


# Time taken for feature extraction
training_feature_extraction_seconds = (time.perf_counter() - training_feature_extraction_start)


if training_features.shape != (5600, FEATURE_DIMENSION):
    raise RuntimeError("Unexpected Training feature matrix shape: "f"{training_features.shape}")


if not np.array_equal(extracted_training_class_indices, training_class_indices):
    raise RuntimeError("Extracted Training class indices do not match the original Training class indices.")


if not np.isfinite(training_features).all():
    raise RuntimeError("Non-finite values were found in the Training feature matrix.")


print("Training images:", training_features.shape[0])
print("Features per image:", training_features.shape[1])
print("Feature matrix shape:", training_features.shape)
print("Class-index alignment verified:", True)
print("Training feature extraction time:", f"{training_feature_extraction_seconds:.2f}s")


Training images: 5600
Features per image: 2048
Feature matrix shape: (5600, 2048)
Class-index alignment verified: True
Training feature extraction time: 24.04s


In [ ]:
# ============================================================
# SAVING THE TRAINING RESNET50 FEATURES
# ============================================================

# Create the persistent hybrid-model results directory
HYBRID_RESULTS_DIR = (
    Path("/content/drive/MyDrive/brain_tumour_colab")
    / "results"
    / "resnet50_fixed_features")

HYBRID_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# Create the path for the cached Training features
TRAINING_FEATURES_PATH = (HYBRID_RESULTS_DIR / "training_resnet50_features.npz")


# Save the features together with their labels, folds,
# and relative image paths
np.savez_compressed(
    TRAINING_FEATURES_PATH,
    features=training_features,
    class_indices=training_class_indices,
    fold_assignments=training_fold_assignments,
    relative_paths=training_relative_paths,
)


print("Training feature cache saved:", TRAINING_FEATURES_PATH)
print("Feature matrix shape:", training_features.shape)

Training feature cache saved: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/training_resnet50_features.npz
Feature matrix shape: (5600, 2048)


In [ ]:
# ============================================================
# DEFINING THE SVM HYPERPARAMETER SEARCH
# ============================================================

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


# Define the SVM pipeline
svm_pipeline = Pipeline(steps=[("scaler", StandardScaler()), ("classifier", SVC())])


# Define the fixed SVM search grid
SVM_PARAMETER_GRID = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__gamma": ["scale", 1e-5, 1e-4, 1e-3, 1e-2]}


# Define the Training-only inner cross-validation
svm_inner_cross_validation = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)


# Create the SVM grid search
svm_grid_search = GridSearchCV(estimator=svm_pipeline, param_grid=SVM_PARAMETER_GRID, scoring="f1_macro", cv=svm_inner_cross_validation,
    n_jobs=-1, refit=True, return_train_score=False)


print("SVM hyperparameter combinations:", 25)
print("Inner cross-validation folds:", 3)
print("Selection metric: macro F1")

SVM hyperparameter combinations: 25
Inner cross-validation folds: 3
Selection metric: macro F1


In [ ]:
# ============================================================
# RUN NESTED FIVE-FOLD CROSS-VALIDATION FOR THE SVM
# ============================================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

svm_nested_cv_start = time.perf_counter()

# Store the outer-fold results
svm_fold_results = []

svm_oof_predictions = np.empty(len(training_class_indices), dtype=np.int32)

# Going through each predefined outer validation fold
for validation_fold in range(1, 6):

    print("=" * 70)
    print(f"SVM OUTER FOLD {validation_fold}")
    print("=" * 70)

    # Identifying the outer Training and validation rows
    outer_training_mask = training_fold_assignments != validation_fold
    outer_validation_mask = training_fold_assignments == validation_fold


    # Create the outer Training data
    outer_training_features = training_features[outer_training_mask]
    outer_training_class_indices = training_class_indices[outer_training_mask]


    # Create the outer validation data
    outer_validation_features = training_features[outer_validation_mask]
    outer_validation_class_indices = training_class_indices[outer_validation_mask]


    print("Outer Training images:", len(outer_training_class_indices))
    print("Outer validation images:", len(outer_validation_class_indices))


    # Run the 3-fold Training-only SVM hyperparameter search
    svm_grid_search.fit(outer_training_features, outer_training_class_indices)


    # Predict the held-out outer validation fold
    fold_predictions = svm_grid_search.predict(outer_validation_features)


    # Store the predictions in their original Training-row positions
    svm_oof_predictions[outer_validation_mask] = fold_predictions


    # Calculate outer-fold metrics
    fold_accuracy = accuracy_score(outer_validation_class_indices, fold_predictions)

    fold_macro_precision = precision_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    fold_macro_recall = recall_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    fold_macro_f1 = f1_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)


    # Save the result for this outer fold
    svm_fold_results.append(
        {
            "fold": validation_fold,
            "accuracy": fold_accuracy,
            "macro_precision": fold_macro_precision,
            "macro_recall": fold_macro_recall,
            "macro_f1": fold_macro_f1,
            "best_C": svm_grid_search.best_params_["classifier__C"],
            "best_gamma": svm_grid_search.best_params_["classifier__gamma"],
            "best_inner_macro_f1": (svm_grid_search.best_score_),
        }
    )


    print("Best parameters:", svm_grid_search.best_params_)
    print("Outer-fold macro F1:", round(fold_macro_f1, 6))

svm_nested_cv_seconds = (time.perf_counter() - svm_nested_cv_start)

print("SVM nested cross-validation time:", f"{svm_nested_cv_seconds:.2f}s")
print("\nSVM five-fold nested cross-validation completed.")

SVM OUTER FOLD 1
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 100, 'classifier__gamma': 0.0001}
Outer-fold macro F1: 0.958303
SVM OUTER FOLD 2
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 100, 'classifier__gamma': 0.0001}
Outer-fold macro F1: 0.944799
SVM OUTER FOLD 3
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 100, 'classifier__gamma': 0.0001}
Outer-fold macro F1: 0.951314
SVM OUTER FOLD 4
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 10, 'classifier__gamma': 'scale'}
Outer-fold macro F1: 0.964308
SVM OUTER FOLD 5
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 10, 'classifier__gamma': 'scale'}
Outer-fold macro F1: 0.953714

SVM five-fold nested cross-validation completed.


In [ ]:
# ======================================================================
# SUMMARIZE THE SVM NESTED CROSS-VALIDATION RESULTS
# ======================================================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix

# Converting the five outer-fold results into a DataFrame
svm_fold_results_dataframe = pd.DataFrame(svm_fold_results)


# Calculating the mean outer-fold metrics
svm_mean_accuracy = svm_fold_results_dataframe["accuracy"].mean()
svm_mean_macro_precision = svm_fold_results_dataframe["macro_precision"].mean()
svm_mean_macro_recall = svm_fold_results_dataframe["macro_recall"].mean()
svm_mean_macro_f1 = svm_fold_results_dataframe["macro_f1"].mean()


# Calculating sample standard deviations across the five folds
svm_std_accuracy = svm_fold_results_dataframe["accuracy"].std(ddof=1)
svm_std_macro_precision = svm_fold_results_dataframe["macro_precision"].std(ddof=1)
svm_std_macro_recall = svm_fold_results_dataframe["macro_recall"].std(ddof=1)
svm_std_macro_f1 = svm_fold_results_dataframe["macro_f1"].std(ddof=1)


# Calculating pooled out-of-fold metrics
svm_oof_accuracy = accuracy_score(training_class_indices, svm_oof_predictions)
svm_oof_macro_precision = precision_score(training_class_indices, svm_oof_predictions, average="macro", zero_division=0)
svm_oof_macro_recall = recall_score(training_class_indices, svm_oof_predictions, average="macro", zero_division=0)
svm_oof_macro_f1 = f1_score(training_class_indices, svm_oof_predictions, average="macro", zero_division=0)


# Create the pooled OOF confusion matrix
svm_oof_confusion_matrix = confusion_matrix(
    training_class_indices,
    svm_oof_predictions,
    labels=list(range(NUMBER_OF_CLASSES)))



# DISPLAY THE RESULTS

display(svm_fold_results_dataframe)
print("\n--- Five-fold mean ± sample SD ---")
print("Accuracy:", f"{svm_mean_accuracy:.6f} ± {svm_std_accuracy:.6f}")
print("Macro precision:",
    f"{svm_mean_macro_precision:.6f} ± "
    f"{svm_std_macro_precision:.6f}")
print("Macro recall:",
    f"{svm_mean_macro_recall:.6f} ± "
    f"{svm_std_macro_recall:.6f}"
print("Macro F1:",
    f"{svm_mean_macro_f1:.6f} ± "
    f"{svm_std_macro_f1:.6f}")
print("\n--- Pooled OOF metrics ---")
print("Accuracy:", f"{svm_oof_accuracy:.6f}")
print("Macro precision:", f"{svm_oof_macro_precision:.6f}")
print("Macro recall:", f"{svm_oof_macro_recall:.6f}")
print("Macro F1:", f"{svm_oof_macro_f1:.6f}")
print("\nPooled OOF confusion matrix:")
print(svm_oof_confusion_matrix)


NameError: name 'svm_fold_results' is not defined

In [ ]:
# ======================================================================
# SELECT THE FINAL SVM HYPERPARAMETERS USING ALL TRAINING FEATURES
# ======================================================================

# Create the final SVM pipeline
final_svm_grid_search = GridSearchCV(estimator=svm_pipeline, param_grid=SVM_PARAMETER_GRID, scoring="f1_macro",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED),
    n_jobs=-1, refit=True, return_train_score=False)


final_svm_search_start = time.perf_counter()

# Running the final training-only hyperparameter search on all 5600 images
final_svm_grid_search.fit(training_features, training_class_indices)

final_svm_search_seconds = (time.perf_counter() - final_svm_search_start)

print("Final SVM hyperparameters:", final_svm_grid_search.best_params_)
print("Best training CV macro F1", round(final_svm_grid_search.best_score_, 6))
print("Final SVM hyperparameter search time:", f"{final_svm_search_seconds:.2f}s")


KeyboardInterrupt: 

In [ ]:
# ======================================================================
# SAVE THE FINAL TRAINED SVM
# ======================================================================

import joblib

# The final SVM already refitted on all 5600 images
final_svm_model = final_svm_grid_search.best_estimator_

# The path for the final trained SVM
FINAL_SVM_MODEL_PATH = (HYBRID_RESULTS_DIR / "final_svm_model.joblib")

# Saving the final SVM
joblib.dump(final_svm_model, FINAL_SVM_MODEL_PATH)

print("Final SVM parameters:", final_svm_grid_search.best_params_)
print("Final SVM model saved:", FINAL_SVM_MODEL_PATH)

AttributeError: 'GridSearchCV' object has no attribute 'best_estimator_'

In [ ]:
# ======================================================================
# DEFINE THE RANDOM FOREST SUCCESSIVE-HALVING SEARCH
# ======================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV, StratifiedKFold

# Original Random Forest search space
RF_PARAMETER_DISTRIBUTIONS = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 20,  40,  60,  80],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", 0.02, 0.05, 0.1, 0.2],
    "bootstrap": [True, False]
}

# Succesive halving configuration
HALVING_INITIAL_CANDIDATES = 100
HALVING_FACTOR = 3
HALVING_MINIMUM_TREES = 25
HALVING_MAXIMUM_TREES = 675

# Random Forest classifier
random_forest_classifier = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1)


# Inner cross-validation
rf_inner_cross_validation = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)


# Create the RF grid search
rf_halving_search = HalvingRandomSearchCV(
    estimator=random_forest_classifier,
    param_distributions=RF_PARAMETER_DISTRIBUTIONS,
    n_candidates=HALVING_INITIAL_CANDIDATES,
    factor=HALVING_FACTOR,
    resource="n_estimators",
    min_resources=HALVING_MINIMUM_TREES,
    max_resources=HALVING_MAXIMUM_TREES,
    scoring="f1_macro",
    cv=rf_inner_cross_validation,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)


print("Initial RF candidates:", HALVING_INITIAL_CANDIDATES)
print("Halving factor:", HALVING_FACTOR)
print("Tree resources: 25 -> 75 -> 225 -> 675")
print("Inner CV folds:", 3)
print("Selection metric: macro F1")
print("Feature standardisation:", False)



Initial RF candidates: 100
Halving factor: 3
Tree resources: 25 -> 75 -> 225 -> 675
Inner CV folds: 3
Selection metric: macro F1
Feature standardisation: False


In [ ]:
HYBRID_RESULTS_DIR = (
    Path("/content/drive/MyDrive/brain_tumour_colab")
    / "results"
    / "resnet50_fixed_features"
)

In [ ]:
# ======================================================================
# RUN NESTED FIVE-FOLD CROSS-VALIDATION FOR THE RANDOM FOREST
# ======================================================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import time

# Create a directory for saving each completed RF fold
RF_CV_RESULTS_DIR = (HYBRID_RESULTS_DIR / "random_forest" / "cross_validation")

RF_CV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# Store the outer-fold results
rf_fold_results = []

rf_oof_predictions = np.empty(len(training_class_indices), dtype=np.int32)

# Start time for the RF evaluation
rf_nested_cv_start = time.perf_counter()

# Going through each predefined outer validation fold
for validation_fold in range(1, 6):

    print("=" * 70)
    print(f"RF OUTER FOLD {validation_fold}")
    print("=" * 70)

    # Start timing the current outer fold
    outer_fold_start = time.perf_counter()


    # Ientifying the outer training and validation rows
    outer_training_mask = training_fold_assignments != validation_fold
    outer_validation_mask = training_fold_assignments == validation_fold


    # Create the outer Training data
    outer_training_features = training_features[outer_training_mask]
    outer_training_class_indices = training_class_indices[outer_training_mask]


    # Create the outer validation data
    outer_validation_features = training_features[outer_validation_mask]
    outer_validation_class_indices = training_class_indices[outer_validation_mask]

    print("Outer Training images:", len(outer_training_class_indices))
    print("Outer validation images:", len(outer_validation_class_indices))

    # Running a 3-fold training only succesive halving search
    rf_halving_search.fit(outer_training_features, outer_training_class_indices)

    # Predict the untouched outer validation fold
    fold_predictions = rf_halving_search.predict(outer_validation_features)

    # Store the predictions in their original Training-row positions
    rf_oof_predictions[outer_validation_mask] = fold_predictions


    # Calculate outer-fold metrics
    fold_accuracy = accuracy_score(outer_validation_class_indices, fold_predictions)

    fold_macro_precision = precision_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    fold_macro_recall = recall_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    fold_macro_f1 = f1_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    # Record the selected hyperparameters
    best_parameters = rf_halving_search.best_params_


    # Calculate the time taken for this outer fold
    outer_fold_seconds = (time.perf_counter() - outer_fold_start)
    print("Outer fold time:", f"{outer_fold_seconds:.2f}s")


    # Storing the selected RF configuration
    fold_results = {
            "fold": validation_fold,
            "accuracy": fold_accuracy,
            "macro_precision": fold_macro_precision,
            "macro_recall": fold_macro_recall,
            "macro_f1": fold_macro_f1,
            "best_n_estimators": (rf_halving_search.best_estimator_.n_estimators),
            "best_criterion": best_parameters["criterion"],
            "best_max_depth": best_parameters["max_depth"],
            "best_min_samples_split": (best_parameters["min_samples_split"]),
            "best_min_samples_leaf": (best_parameters["min_samples_leaf"]),
            "best_max_features": (best_parameters["max_features"]),
            "best_bootstrap": (best_parameters["bootstrap"]),
            "best_inner_macro_f1": (rf_halving_search.best_score_),
            "outer_fold_time": outer_fold_seconds,
            }

    rf_fold_results.append(fold_results)


    # ================================================================
    # SAVE THIS COMPLETED FOLD IMMEDIATELY
    # ================================================================

    fold_result_path = (RF_CV_RESULTS_DIR / f"fold_{validation_fold}_results.csv")
    fold_predictions_path = (RF_CV_RESULTS_DIR / f"fold_{validation_fold}_predictions.npz")


    # Save the fold metrics and selected hyperparameters
    pd.DataFrame([fold_results]).to_csv(fold_result_path, index=False)


    # Save the true labels and predictions for this outer fold
    np.savez_compressed(
        fold_predictions_path,
        true_class_indices=outer_validation_class_indices,
        predicted_class_indices=fold_predictions,
        relative_paths=training_relative_paths[outer_validation_mask])


    print("Best parameters:", rf_halving_search.best_params_)
    print("Selected trees:", rf_halving_search.best_estimator_.n_estimators)
    print("Outer-fold macro F1:", round(fold_macro_f1, 6))
    print("Fold time:", f"{outer_fold_seconds:.2f}s")
    print("Fold saved:", fold_result_path)


# Stop timing
rf_nested_cv_seconds = (time.perf_counter() - rf_nested_cv_start)

print("RF nested cross-validation time:", f"{rf_nested_cv_seconds:.2f}s")
print("\nRF five-fold nested cross-validation completed.")



RF OUTER FOLD 4
Outer Training images: 4480
Outer validation images: 1120
Outer fold time: 7202.23s
Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 0.2, 'max_depth': 40, 'criterion': 'gini', 'bootstrap': False, 'n_estimators': 675}
Selected trees: 675
Outer-fold macro F1: 0.931033
Fold time: 7202.23s
Fold saved: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/random_forest/cross_validation/fold_4_results.csv
RF OUTER FOLD 5
Outer Training images: 4480
Outer validation images: 1120
Outer fold time: 6999.63s
Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.2, 'max_depth': 40, 'criterion': 'gini', 'bootstrap': False, 'n_estimators': 675}
Selected trees: 675
Outer-fold macro F1: 0.928574
Fold time: 6999.63s
Fold saved: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/random_forest/cross_validation/fold_5_results.csv
RF nested cross-validation time: 14201.90s

RF five-fold n

In [ ]:
from sklearn.metrics import confusion_matrix

# ======================================================================
# RECONSTRUCT AND SUMMARIZE ALL FIVE SAVED RF OUTER FOLDS
# ======================================================================


# Store the five saved fold-result tables
saved_rf_fold_results = []

# Reconstruct pooled OOF predictions
reconstructed_rf_oof_predictions = np.empty(len(training_class_indices), dtype=np.int32)

# Track which Training rows receive an OOF prediction
oof_prediction_assigned = np.zeros(len(training_class_indices), dtype=bool)


# Load all five completed folds from Drive
for validation_fold in range(1, 6):

    fold_result_path = RF_CV_RESULTS_DIR / f"fold_{validation_fold}_results.csv"

    fold_predictions_path = RF_CV_RESULTS_DIR / f"fold_{validation_fold}_predictions.npz"


    # Load fold metrics and selected hyperparameters
    fold_result_dataframe = pd.read_csv(fold_result_path)

    saved_rf_fold_results.append(fold_result_dataframe)


    # Load saved validation predictions
    with np.load(fold_predictions_path, allow_pickle=False) as fold_prediction_data:

        fold_true_class_indices = fold_prediction_data["true_class_indices"]

        fold_predicted_class_indices = fold_prediction_data["predicted_class_indices"]

    # Identify the original rows belonging to this validation fold
    outer_validation_mask = training_fold_assignments == validation_fold

    # Verify that the saved labels match the fixed fold assignments
    expected_true_class_indices = training_class_indices[outer_validation_mask]

    if not np.array_equal(fold_true_class_indices, expected_true_class_indices):
        raise RuntimeError("Saved RF labels mismatch.")


    # Restore the saved predictions to their original position
    reconstructed_rf_oof_predictions[outer_validation_mask] = fold_predicted_class_indices

    print(f"Loaded RF fold {validation_fold}:", len(fold_predicted_class_indices), "predictions")

    # Mark these rows as successfully reconstructed
    oof_prediction_assigned[outer_validation_mask] = True


# Combine the five fold-result tables
rf_fold_results_dataframe = pd.concat(saved_rf_fold_results,
    ignore_index=True).sort_values("fold").reset_index(drop=True)

# Verify that five folds were loaded
if len(rf_fold_results_dataframe) != 5:
    raise RuntimeError("Five folds not loaded.")

# Verify that all 5,6000 images received an OOF prediction
if not oof_prediction_assigned.all():
    raise RuntimeError("OOF predictions not assigned.")

# ======================================================================
# FIVE-FOLD MEAN ± SAMPLE STANDARD DEVIATION
# ======================================================================

rf_mean_accuracy = (rf_fold_results_dataframe["accuracy"].mean())

rf_std_accuracy = (rf_fold_results_dataframe["accuracy"].std(ddof=1))

rf_mean_macro_precision = (rf_fold_results_dataframe["macro_precision"].mean())

rf_std_macro_precision = (rf_fold_results_dataframe["macro_precision"].std(ddof=1))

rf_mean_macro_recall = (rf_fold_results_dataframe["macro_recall"].mean())

rf_std_macro_recall = (rf_fold_results_dataframe["macro_recall"].std(ddof=1))

rf_mean_macro_f1 = (rf_fold_results_dataframe["macro_f1"].mean())

rf_std_macro_f1 = (rf_fold_results_dataframe["macro_f1"].std(ddof=1))


# ======================================================================
# POOLED OOF METRICS
# ======================================================================

rf_oof_accuracy = accuracy_score(training_class_indices, reconstructed_rf_oof_predictions)

rf_oof_macro_precision = precision_score(training_class_indices, reconstructed_rf_oof_predictions, average="macro", zero_division=0)

rf_oof_macro_recall = recall_score(training_class_indices, reconstructed_rf_oof_predictions, average="macro", zero_division=0)

rf_oof_macro_f1 = f1_score(training_class_indices, reconstructed_rf_oof_predictions, average="macro", zero_division=0)

rf_oof_confusion_matrix = confusion_matrix(training_class_indices, reconstructed_rf_oof_predictions, labels=list(range(NUMBER_OF_CLASSES)))


# Complete time across all five saved outer folds
rf_total_outer_fold_seconds = (rf_fold_results_dataframe["outer_fold_time"].sum())


# ======================================================================
# DISPLAY RESULTS
# ======================================================================

display(rf_fold_results_dataframe)
print("\n--- Five-fold mean ± sample SD ---")
print("Accuracy:", f"{rf_mean_accuracy:.6f} ± {rf_std_accuracy:.6f}")
print("Macro precision:", f"{rf_mean_macro_precision:.6f} ± "f"{rf_std_macro_precision:.6f}")
print("Macro recall:", f"{rf_mean_macro_recall:.6f} ± "f"{rf_std_macro_recall:.6f}")
print("Macro F1:", f"{rf_mean_macro_f1:.6f} ± {rf_std_macro_f1:.6f}")
print("\n--- Pooled OOF metrics ---")
print("Accuracy:", f"{rf_oof_accuracy:.6f}")
print("Macro precision:", f"{rf_oof_macro_precision:.6f}")
print("Macro recall:", f"{rf_oof_macro_recall:.6f}")
print("Macro F1:", f"{rf_oof_macro_f1:.6f}")
print("\nPooled OOF confusion matrix:")
print(rf_oof_confusion_matrix)
print("\nTotal time across all five outer folds:",f"{rf_total_outer_fold_seconds:.2f}s")
print("All 5,600 OOF predictions reconstructed:",oof_prediction_assigned.all())

Loaded RF fold 1: 1120 predictions
Loaded RF fold 2: 1120 predictions
Loaded RF fold 3: 1120 predictions
Loaded RF fold 4: 1120 predictions
Loaded RF fold 5: 1120 predictions


,fold,accuracy,macro_precision,macro_recall,macro_f1,best_n_estimators,best_criterion,best_max_depth,best_min_samples_split,best_min_samples_leaf,best_max_features,best_bootstrap,best_inner_macro_f1,outer_fold_time
0,1,0.916071,0.920241,0.916071,0.916362,675,gini,NaN,2,1,0.2,False,0.906870,7632.748800
1,2,0.905357,0.907375,0.905357,0.904888,675,gini,40.0,2,1,0.2,False,0.909750,7530.042221
2,3,0.916071,0.918549,0.916071,0.916511,675,entropy,40.0,5,1,0.1,False,0.904863,5639.339466
3,4,0.931250,0.932130,0.931250,0.931033,675,gini,40.0,2,2,0.2,False,0.905929,7202.233179
4,5,0.928571,0.929719,0.928571,0.928574,675,gini,40.0,2,1,0.2,False,0.907286,6999.625524



--- Five-fold mean ± sample SD ---
Accuracy: 0.919464 ± 0.010534
Macro precision: 0.921603 ± 0.009878
Macro recall: 0.919464 ± 0.010534
Macro F1: 0.919474 ± 0.010578

--- Pooled OOF metrics ---
Accuracy: 0.919464
Macro precision: 0.921416
Macro recall: 0.919464
Macro F1: 0.919495

Pooled OOF confusion matrix:
[[1208  175    0   17]
 [  35 1226   49   90]
 [   1   20 1366   13]
 [   9   41    1 1349]]

Total time across all five outer folds: 35003.99s
All 5,600 OOF predictions reconstructed: True


In [ ]:
# ======================================================================
# SELECT THE FINAL RANDOM FOREST HYPERPARAMETERS USING ALL 5,600 TRAINING FEATURES
# ======================================================================

# Create a fresh Random Forest classifier
final_random_forest_classifier = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1)


# Create the final 3-fold Training-only cross-validation
final_rf_inner_cross_validation = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)


# Create a fresh successive-halving search
final_rf_halving_search = HalvingRandomSearchCV(
    estimator=final_random_forest_classifier,
    param_distributions=RF_PARAMETER_DISTRIBUTIONS,
    n_candidates=HALVING_INITIAL_CANDIDATES,
    factor=HALVING_FACTOR,
    resource="n_estimators",
    min_resources=HALVING_MINIMUM_TREES,
    max_resources=HALVING_MAXIMUM_TREES,
    scoring="f1_macro",
    cv=final_rf_inner_cross_validation,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)


# Start timing the final RF search
final_rf_search_start = time.perf_counter()


# Run the final Training-only search on all 5,600 feature vectors
final_rf_halving_search.fit(training_features, training_class_indices)


# Stop timing
final_rf_search_seconds = time.perf_counter() - final_rf_search_start


# Record the refit time for the selected final model
final_rf_refit_seconds = final_rf_halving_search.refit_time_



print("Final RF hyperparameters:", final_rf_halving_search.best_params_)
print("Selected trees:", final_rf_halving_search.best_estimator_.n_estimators)
print("Best Training CV macro F1:", round(final_rf_halving_search.best_score_, 6))
print("Final RF hyperparameter search time:", f"{final_rf_search_seconds:.2f}s")
print("Final RF refit time:", f"{final_rf_refit_seconds:.2f}s")

Final RF hyperparameters: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.2, 'max_depth': None, 'criterion': 'gini', 'bootstrap': False, 'n_estimators': 675}
Selected trees: 675
Best Training CV macro F1: 0.913683
Final RF hyperparameter search time: 10227.56s
Final RF refit time: 3352.03s


In [ ]:
# ======================================================================
# SAVE THE FINAL TRAINED RANDOM FOREST
# ======================================================================

import joblib


# The final RF is already refitted on all 5,600 Training features
final_rf_model = final_rf_halving_search.best_estimator_


# Path for the final trained ResNet50 + RF model
FINAL_RF_MODEL_PATH = HYBRID_RESULTS_DIR / "final_resnet50_random_forest.joblib"


# Save the final RF model
joblib.dump(final_rf_model, FINAL_RF_MODEL_PATH)


print("Final RF parameters:", final_rf_halving_search.best_params_)
print("Final RF model saved:", FINAL_RF_MODEL_PATH)

Final RF parameters: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 0.2, 'max_depth': None, 'criterion': 'gini', 'bootstrap': False, 'n_estimators': 675}
Final RF model saved: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/final_resnet50_random_forest.joblib


In [ ]:
# ======================================================================
# LOAD THE LOCKED FINAL RESNET50 HYBRID CLASSIFIERS
# ======================================================================

import joblib
from pathlib import Path


HYBRID_RESULTS_DIR = Path("/content/drive/MyDrive/brain_tumour_colab") / "results" / "resnet50_fixed_features"


FINAL_SVM_MODEL_PATH = HYBRID_RESULTS_DIR / "final_svm_model.joblib"

FINAL_RF_MODEL_PATH = HYBRID_RESULTS_DIR / "final_resnet50_random_forest.joblib"


# Load the two locked final classifiers
final_svm_model = joblib.load(FINAL_SVM_MODEL_PATH)

final_rf_model = joblib.load(FINAL_RF_MODEL_PATH)


print("Final ResNet50 SVM loaded:", FINAL_SVM_MODEL_PATH)
print("Final ResNet50 RF loaded:", FINAL_RF_MODEL_PATH)

Final ResNet50 SVM loaded: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/final_svm_model.joblib
Final ResNet50 RF loaded: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/final_resnet50_random_forest.joblib


In [ ]:
# ======================================================================
# PREPARING THE HELD-OUT TESTING PARTITION
# ======================================================================

# Testing directory
TESTING_DIR = DATA_DIR / "Testing"

# Collect Testing image paths in the fixed class order
testing_image_paths = []
testing_class_indices = []
testing_relative_paths = []

for class_index, class_name in enumerate(CLASS_NAMES):
    class_directory = TESTING_DIR / class_name

    class_image_paths = sorted(class_directory.glob("*.png"))

    print(f"{class_name} Testing images:", len(class_image_paths))

    for image_path in class_image_paths:
        testing_image_paths.append(image_path)
        testing_class_indices.append(class_index)
        testing_relative_paths.append(str(image_path.relative_to(DATA_DIR)))


# Converting to NumPy arrays
testing_image_paths = np.array(testing_image_paths, dtype = str)
testing_class_indices = np.array(testing_class_indices, dtype = np.int32)
testing_relative_paths = np.array(testing_relative_paths, dtype = str)


# ======================================================================
# VERIFY TESTING PARTITION
# ======================================================================

if len(testing_image_paths) != 1598:
  raise RuntimeError("Wrong number of testing images.")

expected_testing_class_counts = {
    0: 400, # glioma
    1: 400, # meningioma
    2: 398, # notumor
    3: 400, # pituitary

}

for class_index, expected_count in expected_testing_class_counts.items():
    actual_count = np.sum(testing_class_indices == class_index)

    if actual_count != expected_count:
        raise RuntimeError(f"Unexpected Testing count for class {class_index}.")


print("\nTesting images:", len(testing_image_paths))
print("Testing partition verification passed")



glioma Testing images: 400
meningioma Testing images: 400
notumor Testing images: 398
pituitary Testing images: 400

Testing images: 1598
Testing partition verification passed


In [ ]:
# ======================================================================
# CREATE THE RESNET50 FEATURE DATASET FOR THE HELD-OUT TESTING PARTITION
# ======================================================================

FEATURE_EXTRACTION_BATCH_SIZE = 64

testing_feature_dataset = create_resnet50_feature_dataset(
    image_paths = testing_image_paths,
    class_indices = testing_class_indices,
    batch_size = FEATURE_EXTRACTION_BATCH_SIZE)

print("Testing images:", len(testing_image_paths))
print("Feature extraction batch size", FEATURE_EXTRACTION_BATCH_SIZE)
print("Testing feature dataset:", testing_feature_dataset)
print("Testing feature dataset verified")

Testing images: 1598
Feature extraction batch size 64
Testing feature dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
Testing feature dataset verified


In [ ]:
# ======================================================================
# EXTRACT RESNET50 FEATURES FROM THE HELD-OUT TESTING PARTITION
# ======================================================================

# Start timing the feature extraction
testing_feature_extraction_start = time.perf_counter()

# Extract fixed ResNet50 features
testing_features, extracted_testing_class_indices = extract_resnet50_features(
    dataset = testing_feature_dataset,
    feature_extractor = resnet50_feature_extractor)

# Stop the timing
testing_feature_extraction_seconds = time.perf_counter() - testing_feature_extraction_start


# ======================================================================
# VERIFY THE EXTRACTED TESTING FEATURES
# ======================================================================

if testing_features.shape != (len(testing_class_indices), FEATURE_DIMENSION):
    raise RuntimeError("Unexpected testing feature matrix")

if not np.array_equal(extracted_testing_class_indices, testing_class_indices):
    raise RuntimeError("Unexpected testing class indices")

if not np.isfinite(testing_features).all():
    raise RuntimeError("Non-finite testing features")

print("Testing images:", testing_features.shape[0])
print("Features per image:", testing_features.shape[1])
print("Testing feature matrix shape:", testing_features.shape)
print("Testing feature extraction time:", f"{testing_feature_extraction_seconds:.2f}s")
print("Class-index alignment verified:", True)

Testing images: 1598
Features per image: 2048
Testing feature matrix shape: (1598, 2048)
Testing feature extraction time: 6.33s
Class-index alignment verified: True


In [ ]:
# ======================================================================
# FINAL HELD-OUT TESTING EVALUATION: RESNET50 + SVM
# ======================================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Start timing classifier prediction only
svm_testing_prediction_start = time.perf_counter()

# Predict the held-out Testing partition
svm_testing_predictions = final_svm_model.predict(testing_features)

# Stop the timing
svm_testing_prediction_seconds = time.perf_counter() - svm_testing_prediction_start

# End to end testing time: ResNet50 feature extraction + SVM prediction
svm_testing_end_to_end_seconds = testing_feature_extraction_seconds + svm_testing_prediction_seconds


# ======================================================================
# CALCULATE FINAL TESTING METRICS
# ======================================================================

svm_testing_accuracy = accuracy_score(testing_class_indices, svm_testing_predictions)

svm_testing_macro_precision = precision_score(testing_class_indices, svm_testing_predictions, average="macro", zero_division=0)

svm_testing_macro_recall = recall_score(testing_class_indices, svm_testing_predictions, average="macro", zero_division=0)

svm_testing_macro_f1 = f1_score(testing_class_indices, svm_testing_predictions, average="macro", zero_division=0)

svm_testing_confusion_matrix = confusion_matrix(testing_class_indices, svm_testing_predictions, labels=list(range(NUMBER_OF_CLASSES)))

svm_testing_classification_report = classification_report(testing_class_indices, svm_testing_predictions, labels = list(range(NUMBER_OF_CLASSES)), target_names=CLASS_NAMES, zero_division=0, digits = 6)


# ======================================================================
# DISPLAY THE FINAL TESTING RESULTS
# ======================================================================

print("--- ResNet50 + SVM: Held-out Testing results ---")
print("Accuracy:", f"{svm_testing_accuracy:.6f}")
print("Macro precision:", f"{svm_testing_macro_precision:.6f}")
print("Macro recall:", f"{svm_testing_macro_recall:.6f}")
print("Macro F1:", f"{svm_testing_macro_f1:.6f}")
print("\nClassification report:")
print(svm_testing_classification_report)
print("Confusion matrix:")
print(svm_testing_confusion_matrix)
print("\nSVM prediction time:", f"{svm_testing_prediction_seconds:.4f}s")
print("ResNet50 Testing feature extraction time:", f"{testing_feature_extraction_seconds:.2f}s")
print("End-to-end Testing time:", f"{svm_testing_end_to_end_seconds:.2f}s")


--- ResNet50 + SVM: Held-out Testing results ---
Accuracy: 0.936796
Macro precision: 0.941051
Macro recall: 0.936875
Macro F1: 0.935316

Classification report:
              precision    recall  f1-score   support

      glioma   0.975155  0.785000  0.869806       400
  meningioma   0.860619  0.972500  0.913146       400
     notumor   0.940898  1.000000  0.969549       398
   pituitary   0.987531  0.990000  0.988764       400

    accuracy                       0.936796      1598
   macro avg   0.941051  0.936875  0.935316      1598
weighted avg   0.941051  0.936796  0.935273      1598

Confusion matrix:
[[314  61  22   3]
 [  6 389   3   2]
 [  0   0 398   0]
 [  2   2   0 396]]

SVM prediction time: 10.1743s
ResNet50 Testing feature extraction time: 6.33s
End-to-end Testing time: 16.51s


In [ ]:
# ======================================================================
# FINAL HELD-OUT TESTING EVALUATION: RESNET50 + RANDOM FOREST
# ======================================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Start timing classifier prediction only
rf_testing_prediction_start = time.perf_counter()

# Predict the held-out Testing partition
rf_testing_predictions = final_rf_model.predict(testing_features)

# Stop the timing
rf_testing_prediction_seconds = time.perf_counter() - rf_testing_prediction_start

# End to end testing time: ResNet50 feature extraction + RF prediction
rf_testing_end_to_end_seconds = testing_feature_extraction_seconds + rf_testing_prediction_seconds


# ======================================================================
# CALCULATE FINAL TESTING METRICS
# ======================================================================

rf_testing_accuracy = accuracy_score(testing_class_indices, rf_testing_predictions)

rf_testing_macro_precision = precision_score(testing_class_indices, rf_testing_predictions, average="macro", zero_division=0)

rf_testing_macro_recall = recall_score(testing_class_indices, rf_testing_predictions, average="macro", zero_division=0)

rf_testing_macro_f1 = f1_score(testing_class_indices, rf_testing_predictions, average="macro", zero_division=0)

rf_testing_confusion_matrix = confusion_matrix(testing_class_indices, rf_testing_predictions, labels=list(range(NUMBER_OF_CLASSES)))

rf_testing_classification_report = classification_report(testing_class_indices, rf_testing_predictions, labels = list(range(NUMBER_OF_CLASSES)), target_names=CLASS_NAMES, zero_division=0, digits = 6)


# ======================================================================
# DISPLAY THE FINAL TESTING RESULTS
# ======================================================================

print("--- ResNet50 + Random Forest: Held-out Testing results ---")
print("Accuracy:", f"{rf_testing_accuracy:.6f}")
print("Macro precision:", f"{rf_testing_macro_precision:.6f}")
print("Macro recall:", f"{rf_testing_macro_recall:.6f}")
print("Macro F1:", f"{rf_testing_macro_f1:.6f}")
print("\nClassification report:")
print(rf_testing_classification_report)
print("Confusion matrix:")
print(rf_testing_confusion_matrix)
print("\nRF prediction time:", f"{rf_testing_prediction_seconds:.4f}s")
print("ResNet50 Testing feature extraction time:", f"{testing_feature_extraction_seconds:.2f}s")
print("End-to-end Testing time:", f"{rf_testing_end_to_end_seconds:.2f}s")


--- ResNet50 + Random Forest: Held-out Testing results ---
Accuracy: 0.895494
Macro precision: 0.904624
Macro recall: 0.895625
Macro F1: 0.892233

Classification report:
              precision    recall  f1-score   support

      glioma   0.961268  0.682500  0.798246       400
  meningioma   0.785263  0.932500  0.852571       400
     notumor   0.923434  1.000000  0.960193       398
   pituitary   0.948529  0.967500  0.957921       400

    accuracy                       0.895494      1598
   macro avg   0.904624  0.895625  0.892233      1598
weighted avg   0.904600  0.895494  0.892148      1598

Confusion matrix:
[[273  91  30   6]
 [  9 373   3  15]
 [  0   0 398   0]
 [  2  11   0 387]]

RF prediction time: 0.2463s
ResNet50 Testing feature extraction time: 6.33s
End-to-end Testing time: 6.58s


In [ ]:
# ======================================================================
# SAVE THE FINAL RESNET50 HYBRID TESTING RESULTS
# ======================================================================

RESNET50_TESTING_RESULTS_DIR = HYBRID_RESULTS_DIR / "testing"


RESNET50_TESTING_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================
# SAVE SUMMARY METRICS
# ======================================================================

resnet50_testing_summary = pd.DataFrame(
    [
        {
            "classifier": "SVM",
            "accuracy": svm_testing_accuracy,
            "macro_precision": svm_testing_macro_precision,
            "macro_recall": svm_testing_macro_recall,
            "macro_f1": svm_testing_macro_f1,
            "prediction_time_seconds": svm_testing_prediction_seconds,
            "feature_extraction_time_seconds": testing_feature_extraction_seconds,
            "end_to_end_time_seconds": svm_testing_end_to_end_seconds,
        },
        {
            "classifier": "Random Forest",
            "accuracy": rf_testing_accuracy,
            "macro_precision": rf_testing_macro_precision,
            "macro_recall": rf_testing_macro_recall,
            "macro_f1": rf_testing_macro_f1,
            "prediction_time_seconds": rf_testing_prediction_seconds,
            "feature_extraction_time_seconds": testing_feature_extraction_seconds,
            "end_to_end_time_seconds": rf_testing_end_to_end_seconds,
        },
    ]
)


TESTING_SUMMARY_PATH = (RESNET50_TESTING_RESULTS_DIR / "resnet50_hybrid_testing_summary.csv")

resnet50_testing_summary.to_csv(TESTING_SUMMARY_PATH, index=False)


# ======================================================================
# SAVE TESTING PREDICTIONS
# ======================================================================

TESTING_PREDICTIONS_PATH = (RESNET50_TESTING_RESULTS_DIR / "resnet50_hybrid_testing_predictions.npz")

np.savez_compressed(
    TESTING_PREDICTIONS_PATH,
    true_class_indices=testing_class_indices,
    svm_predictions=svm_testing_predictions,
    rf_predictions=rf_testing_predictions,
    relative_paths=testing_relative_paths,
)


# ======================================================================
# SAVE CONFUSION MATRICES
# ======================================================================

np.save(RESNET50_TESTING_RESULTS_DIR / "svm_testing_confusion_matrix.npy", svm_testing_confusion_matrix)

np.save(RESNET50_TESTING_RESULTS_DIR / "rf_testing_confusion_matrix.npy", rf_testing_confusion_matrix)


print("Testing summary saved:", TESTING_SUMMARY_PATH)
print("Testing predictions saved:", TESTING_PREDICTIONS_PATH)
print("ResNet50 hybrid Testing results saved successfully.")

Testing summary saved: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/testing/resnet50_hybrid_testing_summary.csv
Testing predictions saved: /content/drive/MyDrive/brain_tumour_colab/results/resnet50_fixed_features/testing/resnet50_hybrid_testing_predictions.npz
ResNet50 hybrid Testing results saved successfully.
